# WiseOwl four-metric batch run over BioPortal (version 1.1.1)

This notebook scores every public BioPortal ontology with the four WiseOwl core metrics: **Describe, Define, Connection, and Flat**. The reasoner and the three diagnostic metrics are not used.

The metric code is copied from WiseOwl 0.10.0 (github.com/aryand1/WiseOwl) with the same formulas. Step 1 checks this: it runs WiseOwl's own two test ontologies and compares the scores with WiseOwl's stored correct answers.

**How it works, in order:**

1. **Parity check.** Confirms the copied code gives WiseOwl's exact scores on this machine.
2. **Catalog.** Lists all BioPortal ontologies and saves their metadata.
3. **Pilot (optional but recommended).** Runs about 20 ontologies of different sizes to measure time and memory.
4. **Downloads.** Downloads each ontology file (RDF format first).
5. **Evaluation.** Scores each file in a separate background process on the GPU.
6. **Export.** Writes all tables, a data dictionary, and SQL files for Cloudflare D1.

**Resume after a crash.** Every result is saved to `state.sqlite` as soon as it finishes. If Windows restarts, Jupyter crashes, or you stop the kernel, open the notebook again, run the setup cells (up to and including "Start the session"), and then run the step you were on. Finished ontologies are skipped.

**Already ran version 1.0 in `C:\\owl4_run`?** You do not need to start over. Run the setup cells (Settings, API key, the three code-module cells, Start the session) and Step 1, then skip to **Step 6** and **Step 7**. They fix the failures from the first run and re-score only the ontologies that version 1.1 changes. Then run **Step 9** (export).

**What is new in 1.1:**

- **BioPortal-aware Define** (`define_bp_score`, `core_average_bp`): a second Define that also reads each ontology's own definition property, such as NCIT's "DEFINITION" (P97). The strict WiseOwl Define is still reported unchanged. The web page ranks by `core_average_bp`.
- **Repairs:** broken downloads are retried; files rdflib cannot read go through owlready2 (OWL/XML and difficult RDF/XML); OBO-only ontologies such as CHEBI are fetched as OWL from the OBO Foundry; empty files and non-ontology files are labelled `empty` and `not_an_ontology` in place of a 0.00 score.
- **Reproducible Flat:** WiseOwl's Flat can change by 1 point between runs on ontologies with subclass cycles, because Python randomizes set order in each process. The worker now starts with a fixed hash seed, which keeps WiseOwl's exact algorithm and gives the same result every run.
- **1.1.1:** the BioPortal-aware Define ignores `rdfs:isDefinedBy` (it links to a defining document, not definition text) and counts only text values of extra properties. Re-running Step 7 re-scores the ontologies scored under the 1.1 rule.
- **Every parse attempt's error is saved** (`parse_failure.json` in the ontology's results folder).

**Safety limits.** Each ontology runs in a background worker process. If one ontology hangs past the time limit, uses too much RAM, or crashes Python, the notebook kills that worker, records the reason, starts a new worker, and moves on.

## One-time setup (Anaconda Prompt)

Run these commands once in **Anaconda Prompt**, not inside the notebook.

```bat
conda create -n owl4 python=3.12 -y
conda activate owl4

:: PyTorch with CUDA 12.8 or newer. The RTX 50-series (Blackwell) needs this.
:: If pytorch.org lists a newer CUDA build (cu129, cu130, ...), that also works.
pip install torch --index-url https://download.pytorch.org/whl/cu128

pip install "transformers>=4.40" "rdflib>=7.0" owlready2 numpy pandas pyarrow requests psutil jupyter ipykernel
python -m ipykernel install --user --name owl4 --display-name "Python (owl4)"
```

**Updating an environment you already have?** Only one new package is needed:

```bat
conda activate rtx5090_env
pip install owlready2
```

Then start Jupyter from the same prompt (`jupyter lab` or `jupyter notebook`), open this notebook, and choose the kernel **Python (owl4)**.

Also check these before the long run:

- Update the NVIDIA driver to a current version (RTX 50-series cards need a recent driver).
- Set Windows power settings so the PC does not sleep. The notebook also asks Windows to stay awake while it downloads and evaluates, but a manual setting is safer.
- Pick a work folder on a drive with plenty of free space. BioPortal downloads can add up to tens of GB. A short path such as `C:\owl4_run` avoids Windows path-length problems with zip files.
- Get your BioPortal API key from your BioPortal account page (Account, then API Key).

## Settings

Change values here before running. The defaults match WiseOwl's own `config.toml` for the four metrics.

In [ ]:
CONFIG = {
    # Folder for all outputs, downloads and the resume database. Use a short path on a big drive.
    "work_dir": r"C:\owl4_run",

    # BioPortal
    "bioportal_api": "https://data.bioontology.org",
    "requests_per_second": 5.0,        # stay polite to BioPortal
    "include_views": False,            # BioPortal "views" are subsets of other ontologies
    "download_format": "rdf",          # ask BioPortal for RDF/XML (WiseOwl reads RDF only, not OBO)
    "download_fallback_original": True,
    "max_download_gb": None,           # e.g. 5 to skip files bigger than 5 GB; None = no limit
    "keep_downloads": True,
    "download_stream_retries": 3,      # retry a download that breaks part-way (PR did this in 1.0)
    "obo_owl_fallback": True,          # for OBO-only ontologies, try the OBO Foundry OWL file
    "use_owlready2": True,             # second parser for OWL/XML and RDF/XML that rdflib rejects
    "python_hash_seed": 0,             # makes Flat reproducible on ontologies with subclass cycles

    # GPU and model (Define metric)
    "device": "auto",                  # auto | cuda | cpu
    "bert_name": "bert-base-uncased",
    "bert_max_length": 128,
    "batch_size_gpu": 128,
    "batch_size_cpu": 32,
    "mixed_precision": False,          # keep False: True is faster but can change Define in the last decimal
    "define_vectors_on_gpu_max_gb": 4.0,
    "define_cpu_fallback": True,       # if the GPU runs out of memory, redo Define on the CPU

    # WiseOwl metric settings (same as WiseOwl config.toml)
    "define_min_tokens": 12,
    "depth_target": 5,
    "branch_target": 3,
    "suggestion_threshold": 4.0,

    # Safety limits for each ontology
    "eval_timeout_min": 120,           # kill and record "timeout" after this many minutes
    "worker_ram_limit_gb": None,       # None = 85% of this PC's RAM
    "max_attempts": 2,                 # retries for crashes/errors before "gave_up"

    # Extra outputs
    "save_entity_tables": True,        # one CSV row per class/individual with every per-entity value
    "entity_rows_max": None,           # cap rows per ontology; None = all
    "heartbeat_s": 60,                 # progress line for long ontologies, in seconds
    "keep_awake": True,                # ask Windows not to sleep during downloads and evaluation
}

In [ ]:
# BioPortal API key: read from the BIOPORTAL_API_KEY environment variable, or typed in (hidden).
# The key is never written to any output file.
import os, getpass
BIOPORTAL_API_KEY = os.environ.get("BIOPORTAL_API_KEY") or getpass.getpass("BioPortal API key: ")
print("API key loaded" if BIOPORTAL_API_KEY else "No API key")

## Code modules

The next three cells write three Python files next to this notebook: `owl4_core.py` (the copied WiseOwl metrics), `owl4_worker.py` (the background process), and `owl4_pipeline.py` (BioPortal, resume database, exports). They must be real files because Windows starts background processes by importing them. Run all three cells every time you open the notebook.

In [ ]:
%%writefile owl4_core.py
"""owl4_core: the four WiseOwl core metrics (Describe, Define, Connection, Flat).

Copied from WiseOwl 0.10.0 (https://github.com/aryand1/WiseOwl):
ontology/vocab.py, ontology/index.py, ontology/loader.py, ontology/identity.py,
compute/embeddings.py, metrics/describe.py, define.py, connection.py, flat.py
and the core part of reporting.py.

The score math is unchanged. The additions only record more detail:
per-metric counts, per-entity values, timings, and memory use.

Version 1.1 adds (strict WiseOwl scores are unchanged):
* define_bp: a second Define score that also reads each ontology's own
  definition property (from BioPortal metadata, or detected by name, such as
  NCIT's "DEFINITION" property P97). The strict Define is still reported.
* owlready2 fallback parser for OWL/XML and for RDF/XML that rdflib rejects.
* Every parse attempt's error is recorded.

Two engineering changes, neither of which changes the formulas:
1. Define streams the definition vectors batch by batch and compares each
   batch with its label vectors, so very large ontologies do not need all
   vectors on the GPU at once. Batch order is the same length-sorted order
   WiseOwl uses.
2. The parser tries the format detected from the file content first, then
   falls back to WiseOwl's own format list.
"""

from __future__ import annotations

import gc
import hashlib
import json
import logging
import math
import os
import re
import time
from collections import Counter, defaultdict
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass
from typing import Any, NamedTuple, Union

import numpy as np
import rdflib
from rdflib import BNode, Literal, URIRef
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, Namespace

log = logging.getLogger("owl4.core")

WISEOWL_SOURCE_VERSION = "0.10.0"
CORE4_VERSION = "core4-1.1.1"

Node = Union[URIRef, BNode, Literal]
Pair = tuple

# =============================================================================
# vocab.py (copied)
# =============================================================================

OBOINOWL = Namespace("http://www.geneontology.org/formats/oboInOwl#")
IAO = Namespace("http://purl.obolibrary.org/obo/")
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")
IAO_DEFINITION: URIRef = IAO["IAO_0000115"]

CLASS_TYPES: tuple = (OWL.Class, RDFS.Class)

DESCRIBE_SIGNALS: frozenset = frozenset(
    {
        RDFS.label, SKOS.prefLabel, SKOS.altLabel, RDFS.comment, DCTERMS.description,
        DCTERMS.title, SKOS.definition, IAO_DEFINITION, OBOINOWL.hasExactSynonym,
        OBOINOWL.hasRelatedSynonym, OBOINOWL.hasBroadSynonym, OBOINOWL.hasNarrowSynonym,
    }
)

DEFINE_DEFINITION_PROPS: tuple = (
    RDFS.comment, DCTERMS.description, SKOS.definition, IAO_DEFINITION, OBOINOWL.hasDefinition,
)

LABEL_PROPS: tuple = (SKOS.prefLabel, RDFS.label)

CONNECTION_ANNOTATION_PROPS: frozenset = frozenset(
    {
        RDFS.label, RDFS.comment, RDFS.seeAlso, RDFS.isDefinedBy, DCTERMS.title,
        DCTERMS.creator, DCTERMS.description, DCTERMS.abstract, DCTERMS.subject,
        DCTERMS.identifier, SKOS.prefLabel, SKOS.altLabel, SKOS.hiddenLabel,
        SKOS.definition, SKOS.note, SKOS.scopeNote, SKOS.changeNote, SKOS.editorialNote,
        SKOS.historyNote, OBOINOWL.hasExactSynonym, OBOINOWL.hasRelatedSynonym,
        OBOINOWL.hasBroadSynonym, OBOINOWL.hasNarrowSynonym,
    }
)

STRUCTURAL_PREDICATES: frozenset = frozenset(
    {
        RDF.type, RDFS.subClassOf, RDFS.subPropertyOf, RDFS.domain, RDFS.range,
        OWL.equivalentClass, OWL.disjointWith, OWL.unionOf, OWL.intersectionOf,
        OWL.complementOf, OWL.oneOf, OWL.sameAs, OWL.differentFrom, OWL.hasKey,
        OWL.onProperty, OWL.someValuesFrom, OWL.allValuesFrom, OWL.propertyChainAxiom,
        OWL.inverseOf, OWL.equivalentProperty,
    }
)

RESTRICTION_FILLER_PROPS: tuple = (OWL.someValuesFrom, OWL.allValuesFrom)

DEFINE_STOPWORDS: frozenset = frozenset(
    "a an the and or of for in on with by to from is are was were be been being "
    "this that these those it its as at".split()
)


def local_name(iri: str) -> str:
    if "#" in iri:
        return iri.rsplit("#", 1)[-1]
    if "/" in iri:
        return iri.rsplit("/", 1)[-1]
    return iri


# =============================================================================
# index.py (copied; Python 3.12 "type" aliases replaced for wider compatibility)
# =============================================================================

_EMPTY: tuple = ()


@dataclass(frozen=True, slots=True)
class OntologyIndex:
    triple_count: int
    _spo: Mapping
    _by_predicate: Mapping
    _by_type: Mapping
    classes: frozenset
    named_classes: frozenset
    individuals: frozenset
    annotation_properties: frozenset

    @classmethod
    def from_graph(cls, graph: rdflib.Graph) -> "OntologyIndex":
        spo: dict = {}
        by_predicate: dict = {}
        by_type: dict = {}
        count = 0
        for s, p, o in graph:
            count += 1
            spo.setdefault(s, {}).setdefault(p, []).append(o)
            by_predicate.setdefault(p, []).append((s, o))
            if p == RDF.type:
                by_type.setdefault(o, set()).add(s)

        frozen_spo = {s: {p: tuple(os_) for p, os_ in ps.items()} for s, ps in spo.items()}
        del spo
        frozen_by_predicate = {p: tuple(pairs) for p, pairs in by_predicate.items()}
        del by_predicate
        frozen_by_type = {t: frozenset(ss) for t, ss in by_type.items()}

        classes = _collect_classes(frozen_by_type, frozen_by_predicate)
        individuals = frozenset(
            s for s, o in frozen_by_predicate.get(RDF.type, _EMPTY)
            if o in classes and s not in classes
        )
        return cls(
            triple_count=count,
            _spo=frozen_spo,
            _by_predicate=frozen_by_predicate,
            _by_type=frozen_by_type,
            classes=classes,
            named_classes=classes - {OWL.Thing, OWL.Nothing},
            individuals=individuals,
            annotation_properties=frozen_by_type.get(OWL.AnnotationProperty, frozenset()),
        )

    def objects(self, subject, predicate) -> tuple:
        return self._spo.get(subject, {}).get(predicate, _EMPTY)

    def predicates(self, subject) -> Iterable:
        return self._spo.get(subject, {}).keys()

    def has(self, subject, predicate) -> bool:
        return predicate in self._spo.get(subject, {})

    def pairs(self, predicate) -> tuple:
        return self._by_predicate.get(predicate, _EMPTY)

    def subjects_of_type(self, rdf_type) -> frozenset:
        return self._by_type.get(rdf_type, frozenset())

    def predicate_groups(self) -> Iterator:
        yield from self._by_predicate.items()

    def rdf_list(self, head) -> list:
        items: list = []
        seen: set = set()
        node = head
        while node != RDF.nil and node not in seen:
            seen.add(node)
            firsts = self.objects(node, RDF.first)
            if not firsts:
                break
            items.append(firsts[0])
            rests = self.objects(node, RDF.rest)
            if not rests:
                break
            node = rests[0]
        return items

    @property
    def entities(self) -> frozenset:
        return self.classes | self.individuals

    def texts(self, subject, predicate) -> list:
        return [str(o) for o in self.objects(subject, predicate)]

    def first_label(self, entity) -> str:
        for prop in LABEL_PROPS:
            values = self.texts(entity, prop)
            if values:
                return values[0]
        return str(entity).split("#")[-1].split("/")[-1]

    def definition(self, entity, props: tuple, *, strip_each: bool) -> str:
        parts: list = []
        for prop in props:
            for value in self.texts(entity, prop):
                if strip_each:
                    text = value.strip()
                    if text:
                        parts.append(text)
                else:
                    parts.append(value)
        joined = " ".join(parts)
        return joined if strip_each else joined.strip()


def _collect_classes(by_type: Mapping, by_predicate: Mapping) -> frozenset:
    classes: set = set()
    for class_type in (*CLASS_TYPES, SKOS.Concept):
        classes.update(s for s in by_type.get(class_type, ()) if isinstance(s, URIRef))
    for child, parent in by_predicate.get(RDFS.subClassOf, ()):
        if isinstance(child, URIRef):
            classes.add(child)
        if isinstance(parent, URIRef):
            classes.add(parent)
    return frozenset(classes)


# =============================================================================
# loader.py (copied) + content sniffing
# =============================================================================

PARSE_FORMATS: Sequence = (None, "xml", "turtle", "n3", "nt", "trig", "trix")
RDFLIB_FORMATS = {"xml", "turtle", "n3", "nt", "trig", "trix", "json-ld", "nquads"}
UNSUPPORTED_FORMATS = {"obo", "ofn", "omn", "html", "zip", "gzip", "empty", "office_document"}
NOT_AN_ONTOLOGY_FORMATS = {"html", "office_document"}


class OntologyParseError(ValueError):
    def __init__(self, message: str, attempts: list | None = None) -> None:
        super().__init__(message)
        self.attempts = attempts or []


def parse_with_owlready2(path: str, kind: str) -> rdflib.Graph:
    """Convert RDF/XML or OWL/XML with owlready2's standalone parsers (no imports loaded)."""
    if kind == "owlxml":
        from owlready2.owlxml_2_ntriples import parse as o2_parse
    else:
        from owlready2.rdfxml_2_ntriples import parse as o2_parse
    graph = rdflib.Graph()
    add = graph.add
    bnodes: dict = {}

    def node(x: str):
        if x.startswith("_:"):
            b = bnodes.get(x)
            if b is None:
                b = bnodes[x] = BNode()
            return b
        return URIRef(x)

    def on_obj(s, p, o):
        add((node(s), URIRef(p), node(o)))

    def on_data(s, p, o, d):
        if d and d.startswith("@"):
            lit = Literal(o, lang=d[1:])
        elif d:
            lit = Literal(o, datatype=URIRef(d))
        else:
            lit = Literal(o)
        add((node(s), URIRef(p), lit))

    with open(path, "rb") as fh:
        o2_parse(fh, on_prepare_obj=on_obj, on_prepare_data=on_data)
    if len(graph) == 0:
        raise ValueError("owlready2 read 0 triples")
    return graph


def sniff_format(path: str, head_bytes: int = 65536) -> str | None:
    """Guess the serialization from the first bytes of the file."""
    with open(path, "rb") as fh:
        raw = fh.read(head_bytes)
    if not raw.strip():
        return "empty"
    if raw[:2] == b"PK":
        try:
            import zipfile
            with zipfile.ZipFile(path) as zf:
                if "[Content_Types].xml" in zf.namelist():
                    return "office_document"
        except Exception:  # noqa: BLE001
            pass
        return "zip"
    if raw[:2] == b"\x1f\x8b":
        return "gzip"
    text = raw.decode("utf-8", errors="ignore").lstrip("\ufeff").lstrip()
    low = text.lower()
    if "<rdf:rdf" in low:
        return "xml"
    if low.startswith("<!doctype html") or low.startswith("<html"):
        return "html"
    if "<ontology" in low and "www.w3.org/2002/07/owl" in low:
        return "owlxml"
    if low.startswith("<?xml"):
        return "trix" if "<trix" in low else "xml"
    if low.startswith("format-version:") or "\n[term]" in low or low.startswith("[term]"):
        return "obo"
    first_lines = [ln.strip() for ln in low.splitlines()[:200] if ln.strip() and not ln.strip().startswith("#")]
    if first_lines and (first_lines[0].startswith("prefix(") or first_lines[0].startswith("ontology(")):
        return "ofn"
    if first_lines and (first_lines[0].startswith("prefix:") or first_lines[0].startswith("ontology:")):
        return "omn"
    if low.startswith("{") or low.startswith("["):
        return "json-ld"
    if any(ln.startswith(("@prefix", "@base", "prefix ", "base ")) for ln in first_lines[:50]):
        return "turtle"
    if first_lines and all(ln.startswith(("<", "_:")) and ln.endswith(".") for ln in first_lines[:20]):
        return "nt"
    return None


def parse_graph(path: str, first_format: str | None = None, *, use_owlready2: bool = True) -> tuple:
    """Parse the file. Returns (graph, format_used, attempts).

    Tries ``first_format`` (from sniffing) first, then WiseOwl's own list,
    then owlready2 (RDF/XML or OWL/XML) if rdflib could not read the file.
    """
    path = os.fspath(path)
    attempts: list = []
    if first_format == "owlxml":
        order: list = []
    else:
        order = []
        if first_format in RDFLIB_FORMATS:
            order.append(first_format)
        order.extend(f for f in PARSE_FORMATS if f not in order)
    last_error: Exception | None = None
    for fmt in order:
        graph = rdflib.Graph()
        t0 = time.perf_counter()
        try:
            graph.parse(path, format=fmt)
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            attempts.append({"format": fmt or "auto", "ok": False,
                             "seconds": round(time.perf_counter() - t0, 3),
                             "error": f"{type(exc).__name__}: {str(exc)[:300]}"})
            del graph
            gc.collect()
            continue
        attempts.append({"format": fmt or "auto", "ok": True,
                         "seconds": round(time.perf_counter() - t0, 3)})
        return graph, fmt or "auto", attempts
    if use_owlready2 and first_format in ("xml", "owlxml", None, "trix"):
        kind = "owlxml" if first_format == "owlxml" else "rdfxml"
        t0 = time.perf_counter()
        try:
            graph = parse_with_owlready2(path, kind)
            attempts.append({"format": f"owlready2:{kind}", "ok": True,
                             "seconds": round(time.perf_counter() - t0, 3)})
            return graph, f"owlready2:{kind}", attempts
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            attempts.append({"format": f"owlready2:{kind}", "ok": False,
                             "seconds": round(time.perf_counter() - t0, 3),
                             "error": f"{type(exc).__name__}: {str(exc)[:300]}"})
    tried = ", ".join(a["format"] for a in attempts)
    raise OntologyParseError(
        f"unsupported format (tried {tried}). Last error: {last_error}", attempts)


# =============================================================================
# identity.py (copied)
# =============================================================================

class OntologyId(NamedTuple):
    ontology_iri: str
    version_iri: str


def file_sha256(path: str, *, length: int | None = 32) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    h = digest.hexdigest()
    return h[:length] if length else h


def ontology_id(graph: rdflib.Graph, path: str) -> OntologyId:
    ontology_iri = None
    version_iri = None
    for subject in graph.subjects(RDF.type, OWL.Ontology):
        ontology_iri = str(subject)
        for version in graph.objects(subject, OWL.versionIRI):
            version_iri = str(version)
        break
    content_hash = None
    if ontology_iri is None or version_iri is None:
        try:
            content_hash = file_sha256(path)
        except OSError:
            content_hash = None
    if ontology_iri is None:
        ontology_iri = f"file:hash:{content_hash}" if content_hash else "anonymous"
    if version_iri is None:
        version_iri = f"filehash:{content_hash}" if content_hash else "filehash:unknown"
    return OntologyId(ontology_iri, version_iri)


# =============================================================================
# embeddings.py (copied ClsEncoder) + streaming paired cosine
# =============================================================================

def _torch():
    import torch
    return torch


class ClsEncoder:
    """Encode texts to [CLS] vectors (copied from WiseOwl compute/embeddings.py)."""

    def __init__(self, tokenizer, model, device, *, max_length: int, batch_size: int,
                 mixed_precision: bool = False, vectors_on_device_max_gb: float = 4.0) -> None:
        torch = _torch()
        self._tokenizer = tokenizer
        self._model = model.to(device).eval()
        self._device = torch.device(device)
        self._max_length = max_length
        self._batch_size = max(1, batch_size)
        self._mixed_precision = mixed_precision
        self._vectors_on_device_max_bytes = int(vectors_on_device_max_gb * (1 << 30))

    @property
    def device(self):
        return self._device

    @property
    def batch_size(self) -> int:
        return self._batch_size

    @property
    def max_length(self) -> int:
        return self._max_length

    def _autocast(self):
        import contextlib
        torch = _torch()
        if self._mixed_precision and self._device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            return torch.autocast(device_type="cuda", dtype=dtype)
        return contextlib.nullcontext()

    def _batches(self, texts: Sequence[str]):
        """Yield (indices, cls_vectors_on_device, token_lengths) in WiseOwl's length-sorted order."""
        torch = _torch()
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
        for start in range(0, len(order), self._batch_size):
            batch_idx = order[start: start + self._batch_size]
            batch = self._tokenizer(
                [texts[i] for i in batch_idx],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self._max_length,
            ).to(self._device)
            with torch.inference_mode(), self._autocast():
                output = self._model(**batch)
            vectors = output.last_hidden_state[:, 0, :].float()
            lengths = batch["attention_mask"].sum(dim=1).cpu().numpy()
            yield batch_idx, vectors, lengths

    def encode(self, texts: Sequence[str]):
        """Same result as WiseOwl ClsEncoder.encode: (n, hidden) float32, input order."""
        torch = _torch()
        hidden = self._model.config.hidden_size
        n = len(texts)
        on_device = n * hidden * 4 <= self._vectors_on_device_max_bytes
        target = self._device if on_device else torch.device("cpu")
        out = torch.empty((n, hidden), dtype=torch.float32, device=target)
        lengths = np.zeros(n, dtype=np.int32)
        for batch_idx, vectors, lens in self._batches(texts):
            idx = torch.tensor(batch_idx, dtype=torch.long, device=target)
            out[idx] = vectors.to(target)
            lengths[batch_idx] = lens
        return out, lengths

    def paired_cosine(self, a_texts: Sequence[str], b_texts: Sequence[str]) -> dict:
        """cosine(CLS(a[i]), CLS(b[i])) for every i, computed on the device.

        Equivalent to WiseOwl's ``rowwise_cosine(encode(a), encode(b))`` but the
        b-vectors are never all held at once.
        """
        torch = _torch()
        import torch.nn.functional as F
        n = len(a_texts)
        t0 = time.perf_counter()
        a_vecs, a_len = self.encode(a_texts)
        t_a = time.perf_counter() - t0
        cos = np.zeros(n, dtype=np.float64)
        b_len = np.zeros(n, dtype=np.int32)
        t1 = time.perf_counter()
        for batch_idx, b_vecs, lens in self._batches(b_texts):
            idx = torch.tensor(batch_idx, dtype=torch.long, device=a_vecs.device)
            a_part = a_vecs[idx].to(self._device)
            sims = F.cosine_similarity(a_part, b_vecs, dim=1)
            cos[batch_idx] = sims.cpu().numpy().astype(np.float64)
            b_len[batch_idx] = lens
        t_b = time.perf_counter() - t1
        return {
            "cosine": cos,
            "a_token_len": a_len,
            "b_token_len": b_len,
            "seconds_encode_a": round(t_a, 3),
            "seconds_encode_b_and_cosine": round(t_b, 3),
            "a_vectors_device": str(a_vecs.device),
        }


def load_encoder(bert_name: str, device: str, *, max_length: int, batch_size: int,
                 mixed_precision: bool, vectors_on_device_max_gb: float) -> ClsEncoder:
    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    model = AutoModel.from_pretrained(bert_name)
    model = model.float()
    return ClsEncoder(tokenizer, model, device, max_length=max_length, batch_size=batch_size,
                      mixed_precision=mixed_precision,
                      vectors_on_device_max_gb=vectors_on_device_max_gb)


# =============================================================================
# Metrics (formulas copied; details added)
# =============================================================================

_WORD = re.compile(r"\w+")
_TRUE_STRINGS = {"true", "1"}


def _short(node) -> str:
    return str(node)


# ---------------------------------------------------------------- Describe

def _has_skosxl_alt_label(index: OntologyIndex, entity) -> bool:
    return any(
        index.has(label_node, SKOSXL.literalForm)
        for label_node in index.objects(entity, SKOSXL.altLabel)
    )


def compute_describe(index: OntologyIndex, entities: list) -> tuple:
    """Describe = 10 * described / entities (WiseOwl metrics/describe.py)."""
    n = len(entities)
    details: dict = {"entities": n}
    if not entities:
        details.update(described=0, reason="no entities")
        return 0.0, details, None
    signals = frozenset(DESCRIBE_SIGNALS) | index.annotation_properties
    described = np.zeros(n, dtype=bool)
    n_signals = np.zeros(n, dtype=np.int32)
    signal_use: Counter = Counter()
    skosxl_only = 0
    for i, entity in enumerate(entities):
        present = [p for p in index.predicates(entity) if p in signals]
        sk = False
        if not present:
            sk = _has_skosxl_alt_label(index, entity)
            if sk:
                skosxl_only += 1
        for p in present:
            signal_use[p] += 1
        n_signals[i] = len(present) + (1 if sk else 0)
        described[i] = bool(present) or sk
    count = int(described.sum())
    score = round(10.0 * count / n, 2)
    builtin = {str(p): signal_use.get(p, 0) for p in sorted(DESCRIBE_SIGNALS, key=str)}
    declared = {str(p): c for p, c in signal_use.most_common() if p not in DESCRIBE_SIGNALS}
    details.update(
        described=count,
        undescribed=n - count,
        described_share=count / n,
        described_by_skosxl_altlabel_only=skosxl_only,
        declared_annotation_properties=len(index.annotation_properties),
        signal_usage_builtin=builtin,
        signal_usage_declared_annotation_properties_top50=dict(list(declared.items())[:50]),
    )
    return score, details, {"described": described, "describe_signal_count": n_signals}


# ---------------------------------------------------------------- Define

def adequacy_parts(text: str, min_tokens: int = 12) -> tuple:
    """Returns (adequacy, completeness, quality, token_count); adequacy as in WiseOwl."""
    tokens = _WORD.findall(text.lower())
    if not tokens:
        return 0.0, 0.0, 0.0, 0
    completeness = min(1.0, len(tokens) / min_tokens)
    quality = 1.0 - sum(t in DEFINE_STOPWORDS for t in tokens) / len(tokens)
    return max(0.0, min(1.0, 0.4 * completeness + 0.6 * quality)), completeness, quality, len(tokens)


def _label_source(index: OntologyIndex, entity) -> str:
    for prop, name in ((SKOS.prefLabel, "skos:prefLabel"), (RDFS.label, "rdfs:label")):
        if index.objects(entity, prop):
            return name
    return "iri_local_name"


def _stats(arr) -> dict:
    if arr is None or len(arr) == 0:
        return {"n": 0}
    a = np.asarray(arr, dtype=np.float64)
    return {
        "n": int(a.size), "mean": float(a.mean()), "std": float(a.std()),
        "min": float(a.min()), "p05": float(np.percentile(a, 5)),
        "p25": float(np.percentile(a, 25)), "median": float(np.median(a)),
        "p75": float(np.percentile(a, 75)), "p95": float(np.percentile(a, 95)),
        "max": float(a.max()),
    }


def _definition_text(index: OntologyIndex, entity, props: tuple, literal_only: frozenset) -> str:
    """Same joining as OntologyIndex.definition(strip_each=False); props in literal_only keep text values only."""
    parts: list = []
    for prop in props:
        for o in index.objects(entity, prop):
            if prop in literal_only and not isinstance(o, Literal):
                continue
            parts.append(str(o))
    return " ".join(parts).strip()


def compute_define(index: OntologyIndex, entities: list, encoder: ClsEncoder, *,
                   min_tokens: int = 12,
                   fallback_encoder: Callable[[], ClsEncoder] | None = None,
                   props: tuple = DEFINE_DEFINITION_PROPS,
                   literal_only: frozenset = frozenset()) -> tuple:
    """Define = 10 * mean(0.4*match + 0.6*adequacy) (WiseOwl metrics/define.py).

    ``props`` is WiseOwl's definition property list for the strict score; the
    BioPortal-aware score passes a longer list.
    """
    n = len(entities)
    details: dict = {"entities": n}
    if not entities:
        details.update(defined=0, reason="no entities")
        return 0.0, details, None

    if literal_only:
        definitions = [_definition_text(index, e, props, literal_only) for e in entities]
    else:
        definitions = [index.definition(e, props, strip_each=False) for e in entities]
    defined_idx = [i for i, d in enumerate(definitions) if d.strip()]
    source_use = Counter()
    for i in defined_idx:
        for prop in props:
            if index.objects(entities[i], prop):
                source_use[str(prop)] += 1
    if not defined_idx:
        details.update(defined=0, reason="no definitions")
        return 0.0, details, {"has_definition": np.zeros(n, dtype=bool)}

    labels = [index.first_label(entities[i]) for i in defined_idx]
    texts = [definitions[i] for i in defined_idx]
    label_sources = Counter(_label_source(index, entities[i]) for i in defined_idx)

    used_device = str(encoder.device)
    fallback_reason = None
    try:
        pc = encoder.paired_cosine(labels, texts)
    except Exception as exc:  # noqa: BLE001
        is_oom = "out of memory" in str(exc).lower() or type(exc).__name__ == "OutOfMemoryError"
        if not (is_oom and fallback_encoder is not None):
            raise
        torch = _torch()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log.warning("Define: GPU out of memory, retrying on CPU")
        fallback_reason = f"{type(exc).__name__}: {str(exc)[:200]}"
        enc2 = fallback_encoder()
        used_device = str(enc2.device)
        pc = enc2.paired_cosine(labels, texts)

    cosines = pc["cosine"]
    mu = float(cosines.mean())
    sd = float(cosines.std())
    sd_raw = sd
    if sd == 0.0:
        sd = 1.0
    match = 1.0 / (1.0 + np.exp(-(cosines - mu) / sd))
    parts = [adequacy_parts(t, min_tokens) for t in texts]
    adequacy = np.fromiter((p[0] for p in parts), dtype=np.float64, count=len(parts))
    completeness = np.fromiter((p[1] for p in parts), dtype=np.float64, count=len(parts))
    quality = np.fromiter((p[2] for p in parts), dtype=np.float64, count=len(parts))
    word_tokens = np.fromiter((p[3] for p in parts), dtype=np.int64, count=len(parts))

    per_entity = np.zeros(n, dtype=np.float64)
    per_entity[defined_idx] = 0.4 * match + 0.6 * adequacy
    score = round(10.0 * float(per_entity.mean()), 2)

    max_len = encoder.max_length
    details.update(
        defined=len(defined_idx),
        undefined=n - len(defined_idx),
        defined_share=len(defined_idx) / n,
        mean_value_over_defined=float((0.4 * match + 0.6 * adequacy).mean()),
        cosine_mean_used_for_zscore=mu,
        cosine_std_used_for_zscore=sd_raw,
        cosine_stats=_stats(cosines),
        match_stats=_stats(match),
        adequacy_stats=_stats(adequacy),
        completeness_stats=_stats(completeness),
        quality_stats=_stats(quality),
        definition_word_tokens_stats=_stats(word_tokens),
        definitions_shorter_than_min_tokens=int((word_tokens < min_tokens).sum()),
        definitions_truncated_at_bert_max_length=int((pc["b_token_len"] >= max_len).sum()),
        labels_truncated_at_bert_max_length=int((pc["a_token_len"] >= max_len).sum()),
        bert_max_length=max_len,
        min_tokens=min_tokens,
        label_source_counts=dict(label_sources),
        definition_source_usage=dict(source_use),
        device=used_device,
        gpu_oom_fallback=fallback_reason,
        batch_size=encoder.batch_size,
        label_vectors_device=pc["a_vectors_device"],
        seconds_encode_labels=pc["seconds_encode_a"],
        seconds_encode_definitions_and_cosine=pc["seconds_encode_b_and_cosine"],
    )
    ent = {
        "has_definition": np.zeros(n, dtype=bool),
        "definition_chars": np.zeros(n, dtype=np.int64),
        "definition_word_tokens": np.zeros(n, dtype=np.int64),
        "definition_bert_tokens": np.zeros(n, dtype=np.int32),
        "label_bert_tokens": np.zeros(n, dtype=np.int32),
        "define_cosine": np.full(n, np.nan),
        "define_match": np.full(n, np.nan),
        "define_adequacy": np.full(n, np.nan),
        "define_value": per_entity,
    }
    di = np.asarray(defined_idx)
    ent["has_definition"][di] = True
    ent["definition_chars"][di] = [len(t) for t in texts]
    ent["definition_word_tokens"][di] = word_tokens
    ent["definition_bert_tokens"][di] = pc["b_token_len"]
    ent["label_bert_tokens"][di] = pc["a_token_len"]
    ent["define_cosine"][di] = cosines
    ent["define_match"][di] = match
    ent["define_adequacy"][di] = adequacy
    return score, details, ent


_DEFINITION_NAMES = {"definition", "def", "defn", "textualdefinition", "definitions"}


def detect_definition_properties(index: OntologyIndex) -> dict:
    """Properties named or labelled "definition" (e.g. NCIT P97, efo:definition) outside WiseOwl's list."""
    found: dict = {}
    for p, pairs in index.predicate_groups():
        if p in DEFINE_DEFINITION_PROPS or not isinstance(p, URIRef):
            continue
        names = [local_name(str(p))] + index.texts(p, RDFS.label) + index.texts(p, SKOS.prefLabel)
        norm = {re.sub(r"[^a-z]", "", n.lower()) for n in names}
        if not norm & _DEFINITION_NAMES:
            continue
        sample = pairs[:1000]
        literals = sum(1 for _, o in sample if isinstance(o, Literal))
        if sample and literals >= 0.8 * len(sample):
            found[p] = len(pairs)
    return found


# Properties that point to where a term is defined, not to definition text.
NOT_DEFINITION_TEXT = frozenset({RDFS.isDefinedBy, RDFS.seeAlso, OWL.sameAs})


def bp_definition_props(index: OntologyIndex, extra_iris: Iterable | None) -> tuple:
    """WiseOwl's list + BioPortal's declared definition property + detected ones.

    Returns (props, sources). Extra properties only contribute text values
    (see compute_define literal_only); rdfs:isDefinedBy and similar are ignored.
    """
    props = list(DEFINE_DEFINITION_PROPS)
    sources: dict = {}
    for iri in extra_iris or ():
        u = URIRef(str(iri))
        if u in NOT_DEFINITION_TEXT:
            sources[str(u)] = "ignored: links to a defining document, not definition text"
            continue
        if u not in props:
            props.append(u)
            sources[str(u)] = "bioportal_definitionProperty"
    for p, n in detect_definition_properties(index).items():
        if p not in props:
            props.append(p)
            sources[str(p)] = f"detected_by_name ({n} uses)"
        elif str(p) in sources:
            sources[str(p)] += " + detected_by_name"
    return tuple(props), sources


# ---------------------------------------------------------------- Connection

_MAX_DISTINCT_PROPS = 5.0
_RICHNESS_LOG_BASE = 11


def _object_properties(index: OntologyIndex) -> tuple:
    annotation_props = frozenset(CONNECTION_ANNOTATION_PROPS) | index.annotation_properties
    object_like: set = set()
    for predicate, pairs in index.predicate_groups():
        if predicate in STRUCTURAL_PREDICATES or predicate in annotation_props:
            continue
        if any(not isinstance(o, Literal) for _, o in pairs):
            object_like.add(predicate)
    declared = index.subjects_of_type(OWL.ObjectProperty)
    result = frozenset((declared | object_like) - annotation_props)
    return result, declared, object_like


def compute_connection(index: OntologyIndex, entities: list) -> tuple:
    """Connection = 10*(0.7 coverage + 0.2 diversity + 0.1 richness) (metrics/connection.py)."""
    n = len(entities)
    details: dict = {"entities": n}
    if not entities:
        details.update(reason="no entities")
        return 0.0, details, None
    entity_set = index.entities
    object_properties, declared, object_like = _object_properties(index)
    distinct_props: dict = defaultdict(set)
    link_counts: dict = defaultdict(int)
    prop_links: Counter = Counter()
    triple_links = 0
    for prop in object_properties:
        for subject, obj in index.pairs(prop):
            if subject in entity_set:
                distinct_props[subject].add(prop)
                link_counts[subject] += 1
                prop_links[prop] += 1
                triple_links += 1
            if obj in entity_set:
                distinct_props[obj].add(prop)
                link_counts[obj] += 1
                prop_links[prop] += 1
                triple_links += 1

    children_of: dict = defaultdict(list)
    for child, parent in index.pairs(RDFS.subClassOf):
        children_of[parent].append(child)
    restriction_links = 0
    restrictions_on_object_props = 0
    for restriction in index.subjects_of_type(OWL.Restriction):
        for prop in index.objects(restriction, OWL.onProperty):
            if prop not in object_properties:
                continue
            restrictions_on_object_props += 1
            for cls in children_of.get(restriction, ()):
                if cls in index.classes:
                    distinct_props[cls].add(prop)
                    link_counts[cls] += 1
                    prop_links[prop] += 1
                    restriction_links += 1

    connected = sum(1 for e in entities if distinct_props.get(e))
    coverage = connected / n
    diversity = sum(min(len(distinct_props.get(e, ())) / _MAX_DISTINCT_PROPS, 1.0) for e in entities) / n
    richness = sum(
        min(math.log(link_counts[e] + 1, _RICHNESS_LOG_BASE), 1.0) if link_counts.get(e) else 0.0
        for e in entities
    ) / n
    score = round(10.0 * (0.7 * coverage + 0.2 * diversity + 0.1 * richness), 2)
    details.update(
        coverage=coverage, diversity=diversity, richness=richness,
        connected_entities=connected, total_entities=n,
        object_properties_used=len(object_properties),
        object_properties_declared=len(declared),
        object_like_predicates_detected=len(object_like),
        links_from_triples=triple_links,
        links_from_restrictions=restriction_links,
        restrictions_on_object_properties=restrictions_on_object_props,
        owl_restrictions_total=len(index.subjects_of_type(OWL.Restriction)),
        top_properties_by_links=[{"property": str(p), "links": c} for p, c in prop_links.most_common(30)],
    )
    ent = {
        "connection_links": np.fromiter((link_counts.get(e, 0) for e in entities), dtype=np.int64, count=n),
        "connection_distinct_properties": np.fromiter(
            (len(distinct_props.get(e, ())) for e in entities), dtype=np.int64, count=n),
    }
    return score, details, ent


# ---------------------------------------------------------------- Flat

def _restriction_fillers(index: OntologyIndex, node) -> Iterator:
    for prop in RESTRICTION_FILLER_PROPS:
        yield from index.objects(node, prop)


def _expression_parents(index: OntologyIndex, expression) -> Iterator:
    yield from _restriction_fillers(index, expression)
    for list_head in index.objects(expression, OWL.intersectionOf):
        for item in index.rdf_list(list_head):
            if isinstance(item, BNode):
                yield from _restriction_fillers(index, item)
            else:
                yield item


def build_taxonomy(index: OntologyIndex) -> dict:
    classes = index.classes
    parent_to_children: dict = {}

    def add_edge(parent, child) -> None:
        if parent in classes and child in classes and parent != OWL.Thing:
            parent_to_children.setdefault(parent, set()).add(child)

    for child, parent in index.pairs(RDFS.subClassOf):
        if isinstance(parent, BNode):
            for implied in _expression_parents(index, parent):
                add_edge(implied, child)
        else:
            add_edge(parent, child)
    for cls in classes:
        for expression in index.objects(cls, OWL.equivalentClass):
            if isinstance(expression, BNode):
                for implied in _expression_parents(index, expression):
                    add_edge(implied, cls)
            else:
                add_edge(expression, cls)
    for parent, children in parent_to_children.items():
        children.discard(parent)
    return parent_to_children


def max_taxonomy_depth(parent_to_children: dict, deterministic: bool = False) -> tuple:
    """Longest downward path (in nodes), cycle-safe. Also returns back-edge count.

    WiseOwl visits classes in Python set order, which changes between processes,
    so an ontology with subclass cycles can get a different depth on each run.
    The batch worker fixes this by starting Python with a fixed hash seed, which
    keeps WiseOwl's exact algorithm (default deterministic=False). deterministic=True
    visits classes in sorted IRI order instead; it is stable but can differ from
    WiseOwl on ontologies with cycles, so it is off by default.
    """
    depth: dict = {}
    back_edges = [0]

    def depth_of(root) -> int:
        if root in depth:
            return depth[root]
        stack: list = [(root, False)]
        on_stack: set = set()
        while stack:
            node, children_done = stack.pop()
            if children_done:
                depth[node] = 1 + max((depth.get(c, 1) for c in parent_to_children.get(node, ())), default=0)
                on_stack.discard(node)
                continue
            if node in depth:
                continue
            if node in on_stack:
                depth[node] = 1
                back_edges[0] += 1
                continue
            on_stack.add(node)
            children = parent_to_children.get(node, ())
            if not children:
                depth[node] = 1
                on_stack.discard(node)
                continue
            stack.append((node, True))
            if deterministic:
                stack.extend((c, False) for c in sorted(children, key=str, reverse=True) if c not in depth)
            else:
                stack.extend((c, False) for c in children if c not in depth)
        return depth.get(root, 1)

    parents = sorted(parent_to_children, key=str) if deterministic else parent_to_children
    max_depth = max((depth_of(p) for p in parents), default=0)
    return max_depth, back_edges[0]


def compute_flat(index: OntologyIndex, entities: list, *, depth_target: int = 5,
                 branch_target: int = 3, deterministic: bool = False) -> tuple:
    """Flat = round((depth score + breadth score) / 2) (metrics/flat.py)."""
    taxonomy = build_taxonomy(index)
    max_depth, back_edges = max_taxonomy_depth(taxonomy, deterministic)
    avg_branch = sum(len(c) for c in taxonomy.values()) / len(taxonomy) if taxonomy else 0.0
    depth_score = min(max_depth / depth_target, 1.0) * 10
    breadth_score = min(avg_branch / branch_target, 1.0) * 10
    score = int(round((depth_score + breadth_score) / 2))
    children_counts = [len(c) for c in taxonomy.values()]
    all_children = set()
    for c in taxonomy.values():
        all_children |= c
    roots = [p for p in taxonomy if p not in all_children]
    details = dict(
        max_depth=max_depth, avg_branching=avg_branch, depth_score=depth_score,
        breadth_score=breadth_score, depth_target=depth_target, branch_target=branch_target,
        parents_with_children=len(taxonomy), taxonomy_edges=sum(children_counts),
        classes_in_taxonomy=len(set(taxonomy) | all_children),
        root_parents=len(roots), cycle_back_edges_cut=back_edges,
        children_per_parent_stats=_stats(children_counts),
        classes_total=len(index.classes),
    )
    n = len(entities)
    ent = {"taxonomy_children": np.fromiter((len(taxonomy.get(e, ())) for e in entities),
                                            dtype=np.int64, count=n)}
    return score, details, ent


# =============================================================================
# reporting.py (core part, copied wording)
# =============================================================================

CORE_LABELS = {"describe_score": "Describe", "define_score": "Define",
               "connection_score": "Connection", "flat_score": "Flat"}
_CORE_REASONS = {
    "describe_score": "the annotation properties and labels do not have a relatable description",
    "define_score": "the annotations lack semantic meaning",
    "connection_score": "the entities have loose connections",
    "flat_score": "the ontology lacks a clear hierarchical structure",
}


def suggestions(scores: dict, threshold: float = 4.0) -> list:
    out = []
    for key, label in CORE_LABELS.items():
        value = scores.get(key)
        if value is not None and value < threshold:
            out.append(f"{label} Score is low ({value:.2f}/10) because {_CORE_REASONS[key]}.")
    return out


def summary_banner(scores: dict) -> str | None:
    present = {k: v for k, v in scores.items() if v is not None}
    if not present:
        return None
    weakest_key, weakest_value = min(present.items(), key=lambda kv: kv[1])
    weakest = CORE_LABELS.get(weakest_key, weakest_key)
    avg = sum(present.values()) / len(present)
    if avg >= 8.0:
        return (f"This ontology scores well across all dimensions (average {avg:.2f}/10). "
                f"The main area to improve is {weakest} ({weakest_value:.2f}/10).")
    if avg >= 5.0:
        return (f"This ontology has mixed quality (average {avg:.2f}/10). "
                f"The weakest dimension is {weakest} ({weakest_value:.2f}/10).")
    return (f"This ontology scores low overall (average {avg:.2f}/10). "
            f"Several dimensions need work, starting with {weakest} ({weakest_value:.2f}/10).")


# =============================================================================
# Ontology-level statistics (extra, no effect on scores)
# =============================================================================

def ontology_stats(index: OntologyIndex, entities: list) -> dict:
    type_counts = {str(t): len(s) for t, s in index._by_type.items()}
    top_types = dict(sorted(type_counts.items(), key=lambda kv: -kv[1])[:30])
    pred_counts = {str(p): len(pairs) for p, pairs in index.predicate_groups()}
    top_preds = dict(sorted(pred_counts.items(), key=lambda kv: -kv[1])[:40])
    lang = Counter()
    for prop in (*LABEL_PROPS, *DEFINE_DEFINITION_PROPS):
        for _, o in index.pairs(prop):
            if isinstance(o, Literal):
                lang[o.language or "(none)"] += 1
    deprecated = 0
    for e in entities:
        for o in index.objects(e, OWL.deprecated):
            if str(o).strip().lower() in _TRUE_STRINGS:
                deprecated += 1
                break
    imports = sorted({str(o) for _, o in index.pairs(OWL.imports)})
    return {
        "triple_count": index.triple_count,
        "subjects": len(index._spo),
        "distinct_predicates": len(pred_counts),
        "classes": len(index.classes),
        "named_classes": len(index.named_classes),
        "individuals": len(index.individuals),
        "entities": len(entities),
        "annotation_properties": len(index.annotation_properties),
        "object_properties_declared": len(index.subjects_of_type(OWL.ObjectProperty)),
        "datatype_properties_declared": len(index.subjects_of_type(OWL.DatatypeProperty)),
        "owl_classes_typed": len(index.subjects_of_type(OWL.Class)),
        "rdfs_classes_typed": len(index.subjects_of_type(RDFS.Class)),
        "skos_concepts": len(index.subjects_of_type(SKOS.Concept)),
        "owl_restrictions": len(index.subjects_of_type(OWL.Restriction)),
        "subclassof_triples": len(index.pairs(RDFS.subClassOf)),
        "equivalentclass_triples": len(index.pairs(OWL.equivalentClass)),
        "deprecated_entities": deprecated,
        "owl_imports_count": len(imports),
        "owl_imports": imports[:200],
        "label_and_definition_languages": dict(lang.most_common(20)),
        "top_rdf_types": top_types,
        "top_predicates": top_preds,
    }


# =============================================================================
# One full evaluation
# =============================================================================

class StageTimer:
    def __init__(self, on_stage: Callable[[str], None] | None = None) -> None:
        self.marks: dict = {}
        self.on_stage = on_stage

    def run(self, name: str, fn: Callable[[], Any]) -> Any:
        if self.on_stage:
            self.on_stage(name)
        t0 = time.perf_counter()
        try:
            return fn()
        finally:
            self.marks[name] = round(time.perf_counter() - t0, 3)


def evaluate_file(path: str, encoder: ClsEncoder, *, min_tokens: int = 12, depth_target: int = 5,
                  branch_target: int = 3, suggestion_threshold: float = 4.0,
                  want_entities: bool = True, entity_rows_max: int | None = None,
                  fallback_encoder: Callable[[], ClsEncoder] | None = None,
                  on_stage: Callable[[str], None] | None = None,
                  extra_definition_props: Iterable | None = None,
                  use_owlready2: bool = True) -> dict:
    """Parse, index and score one ontology file with the four core metrics.

    Returns {"summary": flat dict, "details": nested dict, "entities": DataFrame | None}.
    Raises OntologyParseError when the file cannot be parsed.
    """
    timer = StageTimer(on_stage)
    t_total = time.perf_counter()

    sniffed = timer.run("sniff", lambda: sniff_format(path))
    if sniffed in UNSUPPORTED_FORMATS:
        kind = "not_an_ontology" if sniffed in NOT_AN_ONTOLOGY_FORMATS else "unsupported_format"
        raise OntologyParseError(f"{kind}:{sniffed}", [{"format": sniffed, "ok": False, "error": "skipped by sniffing"}])
    graph, fmt, attempts = timer.run("parse", lambda: parse_graph(path, sniffed, use_owlready2=use_owlready2))
    oid = timer.run("identity", lambda: ontology_id(graph, path))
    index = timer.run("index", lambda: OntologyIndex.from_graph(graph))
    del graph
    gc.collect()

    entities = list(index.entities)
    stats = timer.run("stats", lambda: ontology_stats(index, entities))

    describe, d_det, d_ent = timer.run("describe", lambda: compute_describe(index, entities))
    define, df_det, df_ent = timer.run(
        "define", lambda: compute_define(index, entities, encoder, min_tokens=min_tokens,
                                         fallback_encoder=fallback_encoder))
    bp_props, bp_sources = bp_definition_props(index, extra_definition_props)
    extra_props = frozenset(p for p in bp_props if p not in DEFINE_DEFINITION_PROPS)
    if bp_props == tuple(DEFINE_DEFINITION_PROPS):
        define_bp, dbp_det, dbp_ent = define, {"same_as_strict": True, "defined": df_det.get("defined")}, None
        bp_source = "same_as_strict"
    else:
        define_bp, dbp_det, dbp_ent = timer.run(
            "define_bp", lambda: compute_define(index, entities, encoder, min_tokens=min_tokens,
                                                fallback_encoder=fallback_encoder, props=bp_props,
                                                literal_only=extra_props))
        bp_source = "; ".join(f"{k} [{v}]" for k, v in bp_sources.items())
    dbp_det["extra_properties"] = bp_sources
    connection, c_det, c_ent = timer.run("connection", lambda: compute_connection(index, entities))
    flat, f_det, f_ent = timer.run(
        "flat", lambda: compute_flat(index, entities, depth_target=depth_target,
                                     branch_target=branch_target))

    scores = {"describe_score": describe, "define_score": define,
              "connection_score": connection, "flat_score": flat}
    core_average = sum(scores.values()) / 4.0
    core_average_bp = (describe + define_bp + connection + flat) / 4.0
    weakest = min(scores.items(), key=lambda kv: kv[1])[0]
    tips = suggestions(scores, suggestion_threshold)

    entities_df = None
    if want_entities and entities:
        def build_entities():
            import pandas as pd
            n = len(entities)
            limit = n if entity_rows_max is None else min(n, entity_rows_max)
            sel = slice(0, limit)
            cls = index.classes
            cols: dict = {
                "iri": [str(e) for e in entities[sel]],
                "kind": ["class" if e in cls else "individual" for e in entities[sel]],
                "label": [index.first_label(e)[:300] for e in entities[sel]],
                "label_source": [_label_source(index, e) for e in entities[sel]],
            }
            for src in (d_ent, df_ent, c_ent, f_ent):
                if src:
                    for k, v in src.items():
                        cols[k] = np.asarray(v)[sel]
            if dbp_ent:
                cols["has_definition_bp"] = np.asarray(dbp_ent["has_definition"])[sel]
                cols["define_bp_value"] = np.asarray(dbp_ent["define_value"])[sel]
            return pd.DataFrame(cols)
        entities_df = timer.run("entities_table", build_entities)

    timer.marks["total"] = round(time.perf_counter() - t_total, 3)
    summary = {
        "ontology_iri": oid.ontology_iri,
        "version_iri": oid.version_iri,
        "sniffed_format": sniffed,
        "parse_format": fmt,
        "parse_attempts": len(attempts),
        **{k: v for k, v in stats.items() if not isinstance(v, (dict, list))},
        "describe_score": describe,
        "define_score": define,
        "connection_score": connection,
        "flat_score": flat,
        "core_average": core_average,
        "core_average_2dp": round(core_average, 2),
        "weakest_metric": CORE_LABELS[weakest],
        "suggestions": "; ".join(tips) if tips else "No suggestions",
        "summary_banner": summary_banner(scores),
        "describe_described": d_det.get("described"),
        "describe_share": d_det.get("described_share"),
        "define_defined": df_det.get("defined"),
        "define_share": df_det.get("defined_share"),
        "define_cosine_mean": df_det.get("cosine_mean_used_for_zscore"),
        "define_cosine_std": df_det.get("cosine_std_used_for_zscore"),
        "define_match_mean": (df_det.get("match_stats") or {}).get("mean"),
        "define_adequacy_mean": (df_det.get("adequacy_stats") or {}).get("mean"),
        "define_definitions_truncated": df_det.get("definitions_truncated_at_bert_max_length"),
        "define_device": df_det.get("device"),
        "define_gpu_oom_fallback": df_det.get("gpu_oom_fallback"),
        "define_bp_score": define_bp,
        "define_bp_defined": dbp_det.get("defined"),
        "define_bp_source": bp_source,
        "core_average_bp": core_average_bp,
        "core_average_bp_2dp": round(core_average_bp, 2),
        "empty_ontology": len(entities) == 0,
        "connection_coverage": c_det.get("coverage"),
        "connection_diversity": c_det.get("diversity"),
        "connection_richness": c_det.get("richness"),
        "connection_connected": c_det.get("connected_entities"),
        "connection_object_properties": c_det.get("object_properties_used"),
        "connection_links_triples": c_det.get("links_from_triples"),
        "connection_links_restrictions": c_det.get("links_from_restrictions"),
        "flat_max_depth": f_det.get("max_depth"),
        "flat_avg_branching": f_det.get("avg_branching"),
        "flat_depth_score": f_det.get("depth_score"),
        "flat_breadth_score": f_det.get("breadth_score"),
        "flat_parents": f_det.get("parents_with_children"),
        "flat_edges": f_det.get("taxonomy_edges"),
        "flat_cycle_back_edges": f_det.get("cycle_back_edges_cut"),
        **{f"time_{k}": v for k, v in timer.marks.items()},
        "define_seconds_encode_labels": df_det.get("seconds_encode_labels"),
        "define_seconds_encode_definitions": df_det.get("seconds_encode_definitions_and_cosine"),
        "entity_rows_written": 0 if entities_df is None else len(entities_df),
        "wiseowl_source_version": WISEOWL_SOURCE_VERSION,
        "core4_version": CORE4_VERSION,
    }
    details = {
        "scores": scores,
        "core_average": core_average,
        "define_bp_score": define_bp,
        "core_average_bp": core_average_bp,
        "ontology_id": list(oid),
        "parse": {"sniffed_format": sniffed, "format_used": fmt, "attempts": attempts},
        "stats": stats,
        "describe": d_det,
        "define": df_det,
        "define_bp": dbp_det,
        "connection": c_det,
        "flat": f_det,
        "timings_seconds": timer.marks,
        "suggestions": tips,
        "summary_banner": summary_banner(scores),
    }
    return {"summary": summary, "details": details, "entities": entities_df}


def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        f = float(obj)
        return None if math.isnan(f) else f
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    return obj


def dump_json(obj: Any, path: str) -> None:
    tmp = f"{path}.tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(to_jsonable(obj), fh, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)

In [ ]:
%%writefile owl4_worker.py
"""owl4_worker: background process that evaluates ontologies one at a time.

The notebook (parent) starts this process, sends one task at a time, and
watches its memory and run time. If an ontology hangs, uses too much memory,
or crashes Python, the parent kills this process and starts a new one; the
notebook itself keeps running.

Must live in a .py file: Windows starts child processes with "spawn", which
imports the target function by module name.
"""

from __future__ import annotations

import json
import logging
import os
import platform
import sys
import time
import traceback


def _setup_logging(log_dir: str) -> logging.Logger:
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger("owl4")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fh = logging.FileHandler(os.path.join(log_dir, "worker.log"), encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s [pid %(process)d] %(levelname)s %(name)s: %(message)s"))
        logger.addHandler(fh)
    return logger


def _write_stage(path: str, payload: dict) -> None:
    try:
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as fh:
            json.dump(payload, fh)
        os.replace(tmp, path)
    except OSError:
        pass


def _peak_rss_mb() -> dict:
    try:
        import psutil
        mi = psutil.Process().memory_info()
        out = {"rss_mb_now": round(mi.rss / 2**20, 1)}
        peak = getattr(mi, "peak_wset", None)  # Windows only: lifetime peak
        if peak:
            out["process_lifetime_peak_mb"] = round(peak / 2**20, 1)
        return out
    except Exception:  # noqa: BLE001
        return {}


def worker_main(task_q, result_q, cfg: dict) -> None:
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
    log = _setup_logging(cfg["log_dir"])
    logging.getLogger("rdflib").setLevel(logging.ERROR)
    try:
        import torch
        import owl4_core as core

        t0 = time.perf_counter()
        device_spec = cfg.get("device", "auto")
        if device_spec == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            device = device_spec
            if device.startswith("cuda") and not torch.cuda.is_available():
                log.warning("CUDA requested but not available; using CPU")
                device = "cpu"
        if cfg.get("cpu_threads"):
            torch.set_num_threads(int(cfg["cpu_threads"]))
        batch = cfg["batch_size_gpu"] if device.startswith("cuda") else cfg["batch_size_cpu"]
        encoder = core.load_encoder(
            cfg["bert_name"], device, max_length=cfg["bert_max_length"], batch_size=batch,
            mixed_precision=cfg["mixed_precision"],
            vectors_on_device_max_gb=cfg["define_vectors_on_gpu_max_gb"])
        load_s = round(time.perf_counter() - t0, 2)
        info = {
            "type": "ready", "pid": os.getpid(), "device": device, "batch_size": batch,
            "model_load_seconds": load_s, "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "python": sys.version.split()[0], "platform": platform.platform(),
            "python_hash_seed": os.environ.get("PYTHONHASHSEED"),
        }
        log.info("worker ready: %s", info)
        result_q.put(info)
    except Exception as exc:  # noqa: BLE001
        result_q.put({"type": "fatal", "error": f"{type(exc).__name__}: {exc}",
                      "traceback": traceback.format_exc()})
        return

    cpu_encoder_holder: dict = {}

    def cpu_fallback():
        if "enc" not in cpu_encoder_holder:
            cpu_encoder_holder["enc"] = core.load_encoder(
                cfg["bert_name"], "cpu", max_length=cfg["bert_max_length"],
                batch_size=cfg["batch_size_cpu"], mixed_precision=False,
                vectors_on_device_max_gb=1e9)
        return cpu_encoder_holder["enc"]

    while True:
        task = task_q.get()
        if task is None:
            log.info("worker stopping")
            break
        result_q.put(run_task(task, encoder, cpu_fallback if cfg.get("define_cpu_fallback") else None,
                              cfg, log, core, torch))


def run_task(task: dict, encoder, fallback, cfg: dict, log, core, torch) -> dict:
    key = task["key"]
    out_dir = task["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    stage_file = cfg["stage_file"]
    started = time.time()

    def on_stage(name: str) -> None:
        _write_stage(stage_file, {"key": key, "stage": name, "since": time.time(), "started": started})
        log.info("%s: stage %s", key, name)

    on_stage("start")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    base = {"type": "result", "key": key, "pid": os.getpid()}
    try:
        res = core.evaluate_file(
            task["path"], encoder,
            min_tokens=cfg["define_min_tokens"], depth_target=cfg["depth_target"],
            branch_target=cfg["branch_target"], suggestion_threshold=cfg["suggestion_threshold"],
            want_entities=cfg["save_entity_tables"], entity_rows_max=cfg["entity_rows_max"],
            fallback_encoder=fallback, on_stage=on_stage,
            extra_definition_props=task.get("extra_definition_props"),
            use_owlready2=cfg.get("use_owlready2", True))
    except core.OntologyParseError as exc:
        attempts = getattr(exc, "attempts", [])
        try:
            core.dump_json({"task": task, "error": str(exc), "attempts": attempts},
                           os.path.join(out_dir, "parse_failure.json"))
        except Exception:  # noqa: BLE001
            pass
        return {**base, "status": "parse_failed", "error_type": "OntologyParseError",
                "error": str(exc)[:2000], "traceback": json.dumps(attempts)[:4000],
                "stage": "parse", **_peak_rss_mb()}
    except MemoryError as exc:
        return {**base, "status": "memory_error", "error_type": "MemoryError",
                "error": str(exc)[:2000], "traceback": traceback.format_exc()[-4000:], **_peak_rss_mb()}
    except Exception as exc:  # noqa: BLE001
        return {**base, "status": "error", "error_type": type(exc).__name__,
                "error": str(exc)[:2000], "traceback": traceback.format_exc()[-4000:], **_peak_rss_mb()}

    on_stage("write_outputs")
    t_w = time.perf_counter()
    summary = res["summary"]
    details = res["details"]
    if torch.cuda.is_available():
        summary["gpu_peak_allocated_mb"] = round(torch.cuda.max_memory_allocated() / 2**20, 1)
        summary["gpu_peak_reserved_mb"] = round(torch.cuda.max_memory_reserved() / 2**20, 1)
    summary.update(_peak_rss_mb())
    summary["bert_name"] = cfg["bert_name"]
    summary["mixed_precision"] = cfg["mixed_precision"]
    summary["encoder_device"] = str(encoder.device)
    summary["encoder_batch_size"] = encoder.batch_size
    summary["torch_version"] = torch.__version__
    summary["python_hash_seed"] = os.environ.get("PYTHONHASHSEED")
    try:
        import transformers
        summary["transformers_version"] = transformers.__version__
    except Exception:  # noqa: BLE001
        pass
    import rdflib
    summary["rdflib_version"] = rdflib.__version__

    result_path = os.path.join(out_dir, "result.json")
    core.dump_json({"task": {k: v for k, v in task.items() if k != "catalog"},
                    "summary": summary, "details": details}, result_path)
    entities_path = None
    if res["entities"] is not None:
        entities_path = os.path.join(out_dir, "entities.csv.gz")
        tmp = entities_path + ".tmp"
        res["entities"].to_csv(tmp, index=False, compression="gzip")
        os.replace(tmp, entities_path)
    summary["time_write_outputs"] = round(time.perf_counter() - t_w, 3)
    del res
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    on_stage("done")
    return {**base, "status": "done", "summary": summary, "result_path": result_path,
            "entities_path": entities_path}

In [ ]:
%%writefile owl4_pipeline.py
"""owl4_pipeline: BioPortal catalog, downloads, resumable evaluation, and exports.

Everything that happens is written to a SQLite file (state.sqlite) right away,
so the run can stop at any point (crash, restart, Ctrl+C, power loss) and
continue from where it left off when the same cell is run again.
"""

from __future__ import annotations

import datetime as dt
import gzip
import hashlib
import json
import logging
import os
import platform
import queue
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import traceback
import uuid
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import requests

import owl4_core as core

NOTEBOOK_VERSION = "owl4-batch-1.1.1"

DEFAULT_CONFIG: dict = {
    # where everything goes
    "work_dir": str(Path.home() / "owl4_run"),
    # BioPortal
    "provider": "bioportal",
    "bioportal_api": "https://data.bioontology.org",
    "bioportal_ui": "https://bioportal.bioontology.org/ontologies",
    "requests_per_second": 5.0,
    "http_timeout_s": 60,
    "download_read_timeout_s": 600,
    "max_http_retries": 5,
    "include_views": False,
    "download_format": "rdf",
    "download_fallback_original": True,
    "max_download_gb": None,
    "keep_downloads": True,
    "download_stream_retries": 3,
    "obo_owl_fallback": True,
    "use_owlready2": True,
    "python_hash_seed": 0,
    # model and device
    "device": "auto",
    "bert_name": "bert-base-uncased",
    "bert_max_length": 128,
    "batch_size_gpu": 128,
    "batch_size_cpu": 32,
    "mixed_precision": False,
    "cpu_threads": 0,
    "define_vectors_on_gpu_max_gb": 4.0,
    "define_cpu_fallback": True,
    # WiseOwl metric settings (same defaults as WiseOwl config.toml)
    "define_min_tokens": 12,
    "depth_target": 5,
    "branch_target": 3,
    "suggestion_threshold": 4.0,
    # safety limits per ontology
    "eval_timeout_min": 120,
    "worker_ram_limit_gb": None,
    "max_attempts": 2,
    "worker_ready_timeout_s": 1800,
    "heartbeat_s": 60,
    # extra outputs
    "save_entity_tables": True,
    "entity_rows_max": None,
    "keep_awake": True,
}

RETRYABLE_EVAL = {"error", "crashed", "interrupted", "memory_error"}
TERMINAL_EVAL = {"done", "parse_failed", "timeout", "memory_limit", "gave_up", "skipped"}

log = logging.getLogger("owl4.pipeline")


def now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(text))[:120]


# =============================================================================
# State database
# =============================================================================

SCHEMA = """
CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY, value TEXT);
CREATE TABLE IF NOT EXISTS runs(
  run_id TEXT PRIMARY KEY, started_at TEXT, finished_at TEXT, phase TEXT,
  notebook_version TEXT, config_json TEXT, env_json TEXT, result_json TEXT);
CREATE TABLE IF NOT EXISTS catalog(
  acronym TEXT PRIMARY KEY, name TEXT, submission_id INTEGER, status TEXT, error TEXT,
  fetched_at TEXT, api_seconds REAL,
  ontology_json TEXT, submission_json TEXT, metrics_json TEXT);
CREATE TABLE IF NOT EXISTS downloads(
  acronym TEXT, submission_id INTEGER, status TEXT, variant TEXT, http_status INTEGER,
  url TEXT, raw_path TEXT, eval_path TEXT, bytes INTEGER, eval_bytes INTEGER, sha256 TEXT,
  content_type TEXT, server_filename TEXT, sniffed_format TEXT, archive_type TEXT,
  archive_members TEXT, attempts INTEGER DEFAULT 0, started_at TEXT, finished_at TEXT,
  seconds REAL, error TEXT, attempts_log TEXT, run_id TEXT,
  PRIMARY KEY(acronym, submission_id));
CREATE TABLE IF NOT EXISTS evaluations(
  acronym TEXT, submission_id INTEGER, status TEXT, attempts INTEGER DEFAULT 0,
  started_at TEXT, finished_at TEXT, seconds REAL, error_type TEXT, error TEXT,
  traceback TEXT, stage_at_failure TEXT,
  describe_score REAL, define_score REAL, connection_score REAL, flat_score REAL,
  core_average REAL, summary_json TEXT, result_path TEXT, entities_path TEXT,
  parent_peak_rss_mb REAL, timeout_min REAL, run_id TEXT,
  PRIMARY KEY(acronym, submission_id));
CREATE TABLE IF NOT EXISTS events(
  id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, run_id TEXT, phase TEXT,
  acronym TEXT, level TEXT, message TEXT);
CREATE TABLE IF NOT EXISTS search_benchmark(
  id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, run_id TEXT, endpoint TEXT, keyword TEXT,
  http_status INTEGER, seconds REAL, result_count INTEGER, distinct_ontologies INTEGER,
  top_ontologies TEXT, error TEXT);
"""


class State:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.conn = sqlite3.connect(str(path), timeout=60)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("PRAGMA journal_mode=WAL")
        self.conn.execute("PRAGMA synchronous=NORMAL")
        self.conn.executescript(SCHEMA)
        self.conn.commit()

    def q(self, sql: str, args: tuple = ()) -> list:
        return [dict(r) for r in self.conn.execute(sql, args).fetchall()]

    def one(self, sql: str, args: tuple = ()) -> dict | None:
        r = self.conn.execute(sql, args).fetchone()
        return dict(r) if r else None

    def exec(self, sql: str, args: tuple = ()) -> None:
        self.conn.execute(sql, args)
        self.conn.commit()

    def upsert(self, table: str, row: dict, keys: tuple) -> None:
        cols = list(row)
        sql = (f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join('?' * len(cols))}) "
               f"ON CONFLICT({','.join(keys)}) DO UPDATE SET "
               + ",".join(f"{c}=excluded.{c}" for c in cols if c not in keys))
        self.conn.execute(sql, tuple(row[c] for c in cols))
        self.conn.commit()


# =============================================================================
# Context
# =============================================================================

@dataclass
class Ctx:
    cfg: dict
    work: Path
    state: State
    run_id: str
    api_key: str | None = None
    _api: Any = None
    paths: dict = field(default_factory=dict)

    @property
    def api(self) -> "BioPortal":
        if self._api is None:
            if not self.api_key:
                raise RuntimeError("No BioPortal API key. Set it in the API key cell.")
            self._api = BioPortal(self.api_key, self.cfg)
        return self._api

    def event(self, phase: str, acronym: str | None, level: str, message: str) -> None:
        self.state.conn.execute(
            "INSERT INTO events(ts, run_id, phase, acronym, level, message) VALUES (?,?,?,?,?,?)",
            (now(), self.run_id, phase, acronym, level, message[:4000]))
        self.state.conn.commit()
        getattr(log, level.lower() if level.lower() in ("info", "warning", "error") else "info")(
            "[%s] %s: %s", phase, acronym or "-", message)


def init(cfg_overrides: dict | None = None, api_key: str | None = None) -> Ctx:
    cfg = {**DEFAULT_CONFIG, **(cfg_overrides or {})}
    work = Path(cfg["work_dir"]).expanduser().resolve()
    paths = {name: work / name for name in
             ("logs", "raw", "downloads", "results", "exports", "environment")}
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("owl4")
    logger.setLevel(logging.INFO)
    log_file = str(paths["logs"] / "pipeline.log")
    if not any(isinstance(h, logging.FileHandler) and getattr(h, "baseFilename", "") == log_file
               for h in logger.handlers):
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s"))
        logger.addHandler(fh)
    state = State(work / "state.sqlite")
    run_id = dt.datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
    state.exec("INSERT INTO runs(run_id, started_at, phase, notebook_version, config_json) VALUES (?,?,?,?,?)",
               (run_id, now(), "init", NOTEBOOK_VERSION, json.dumps(cfg, default=str)))
    ctx = Ctx(cfg=cfg, work=work, state=state, run_id=run_id, api_key=api_key, paths=paths)
    ctx.event("init", None, "INFO", f"session started, work_dir={work}")
    return ctx


# =============================================================================
# Environment report
# =============================================================================

def environment_report(ctx: Ctx) -> dict:
    env: dict = {"time": now(), "run_id": ctx.run_id, "notebook_version": NOTEBOOK_VERSION,
                 "python": sys.version, "executable": sys.executable,
                 "platform": platform.platform(), "machine": platform.machine(),
                 "processor": platform.processor(), "cpu_count": os.cpu_count(),
                 "conda_env": os.environ.get("CONDA_DEFAULT_ENV")}
    try:
        import psutil
        vm = psutil.virtual_memory()
        env["ram_total_gb"] = round(vm.total / 2**30, 1)
        env["ram_available_gb"] = round(vm.available / 2**30, 1)
        du = shutil.disk_usage(str(ctx.work))
        env["disk_free_gb"] = round(du.free / 2**30, 1)
    except Exception as exc:  # noqa: BLE001
        env["psutil_error"] = str(exc)
    for mod in ("torch", "transformers", "rdflib", "numpy", "pandas", "requests", "psutil", "pyarrow"):
        try:
            env[f"{mod}_version"] = __import__(mod).__version__
        except Exception:  # noqa: BLE001
            env[f"{mod}_version"] = None
    try:
        import torch
        env["cuda_available"] = torch.cuda.is_available()
        env["torch_cuda_build"] = torch.version.cuda
        env["torch_arch_list"] = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            env["gpu_name"] = props.name
            env["gpu_memory_gb"] = round(props.total_memory / 2**30, 1)
            env["gpu_capability"] = f"{props.major}.{props.minor}"
            arch = f"sm_{props.major}{props.minor}"
            env["gpu_arch_supported_by_this_torch"] = arch in env["torch_arch_list"]
            env["bf16_supported"] = torch.cuda.is_bf16_supported()
    except Exception as exc:  # noqa: BLE001
        env["torch_error"] = str(exc)
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                              "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
        env["nvidia_smi"] = out.stdout.strip() or out.stderr.strip()
    except Exception as exc:  # noqa: BLE001
        env["nvidia_smi"] = f"not available: {exc}"
    path = ctx.paths["environment"] / f"environment_{ctx.run_id}.json"
    core.dump_json(env, str(path))
    ctx.state.exec("UPDATE runs SET env_json=? WHERE run_id=?", (json.dumps(env, default=str), ctx.run_id))
    return env


# =============================================================================
# BioPortal client
# =============================================================================

class AuthError(RuntimeError):
    pass


class BioPortal:
    def __init__(self, api_key: str, cfg: dict) -> None:
        self.base = cfg["bioportal_api"].rstrip("/")
        self.cfg = cfg
        self.min_interval = 1.0 / max(0.1, float(cfg["requests_per_second"]))
        self._last = 0.0
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"apikey token={api_key}",
            "Accept": "application/json",
            "User-Agent": f"{NOTEBOOK_VERSION} (WiseOwl batch evaluation)",
        })

    def _wait(self) -> None:
        delay = self._last + self.min_interval - time.monotonic()
        if delay > 0:
            time.sleep(delay)
        self._last = time.monotonic()

    def request(self, path: str, params: dict | None = None, *, stream: bool = False,
                read_timeout: float | None = None) -> tuple:
        """Returns (response or None, info). Retries 429/5xx/network errors."""
        url = path if path.startswith("http") else f"{self.base}{path}"
        info: dict = {"url": url, "params": params or {}, "tries": []}
        retries = int(self.cfg["max_http_retries"])
        for attempt in range(retries + 1):
            self._wait()
            t0 = time.perf_counter()
            try:
                resp = self.session.get(url, params=params, stream=stream,
                                        timeout=(30, read_timeout or self.cfg["http_timeout_s"]))
            except (requests.ConnectionError, requests.Timeout,
                    requests.exceptions.ChunkedEncodingError) as exc:
                info["tries"].append({"error": f"{type(exc).__name__}: {str(exc)[:200]}",
                                      "seconds": round(time.perf_counter() - t0, 3)})
                if attempt < retries:
                    time.sleep(min(60, 2 ** attempt) + random.random())
                    continue
                info["error"] = info["tries"][-1]["error"]
                return None, info
            info["tries"].append({"status": resp.status_code,
                                  "seconds": round(time.perf_counter() - t0, 3)})
            info["status"] = resp.status_code
            info["seconds"] = info["tries"][-1]["seconds"]
            if resp.status_code == 401:
                raise AuthError("BioPortal returned 401: check the API key.")
            if resp.status_code in (429, 500, 502, 503, 504) and attempt < retries:
                wait = resp.headers.get("Retry-After")
                try:
                    wait_s = float(wait) if wait else min(60, 2 ** attempt)
                except ValueError:
                    wait_s = min(60, 2 ** attempt)
                resp.close()
                time.sleep(wait_s + random.random())
                continue
            return resp, info
        return None, info

    def get_json(self, path: str, params: dict | None = None) -> tuple:
        resp, info = self.request(path, params)
        if resp is None:
            return None, info
        if resp.status_code != 200:
            info["body"] = resp.text[:1000]
            return None, info
        try:
            return resp.json(), info
        except ValueError:
            info["body"] = resp.text[:1000]
            info["error"] = "response was not JSON"
            return None, info


def _tail(value: Any) -> str:
    if isinstance(value, dict):
        value = value.get("acronym") or value.get("name") or value.get("@id") or ""
    return str(value).rstrip("/").split("/")[-1]


# =============================================================================
# Phase A: catalog
# =============================================================================

def catalog_phase(ctx: Ctx, refresh: bool = False, only: list | None = None,
                  limit: int | None = None) -> dict:
    """List BioPortal ontologies and store metadata, latest submission, and metrics."""
    api = ctx.api
    params = {"display": "all", "display_context": "false", "display_links": "false"}
    if ctx.cfg["include_views"]:
        params["also_include_views"] = "true"
    listing, info = api.get_json("/ontologies", params)
    if listing is None:
        ctx.event("catalog", None, "WARNING", f"display=all listing failed ({info.get('status')}); retrying plain")
        listing, info = api.get_json("/ontologies", {"display_context": "false", "display_links": "false"})
    if listing is None:
        raise RuntimeError(f"Could not list ontologies: {info}")
    core.dump_json(listing, str(ctx.paths["raw"] / "ontologies_list.json"))
    ctx.event("catalog", None, "INFO", f"{len(listing)} ontologies listed")

    todo = sorted(listing, key=lambda o: o.get("acronym", ""))
    if only:
        wanted = {a.upper() for a in only}
        todo = [o for o in todo if str(o.get("acronym", "")).upper() in wanted]
    if limit:
        todo = todo[:limit]
    done_rows = {r["acronym"] for r in ctx.state.q("SELECT acronym FROM catalog WHERE status IN ('ok','no_submission')")}
    counts: dict = {}
    t_start = time.time()
    for i, onto in enumerate(todo, 1):
        acr = onto.get("acronym")
        if not acr or (not refresh and acr in done_rows):
            counts["skipped_already_done"] = counts.get("skipped_already_done", 0) + 1
            continue
        t0 = time.perf_counter()
        raw_dir = ctx.paths["raw"] / safe_name(acr)
        raw_dir.mkdir(exist_ok=True)
        core.dump_json(onto, str(raw_dir / "ontology.json"))
        status, error, sub_id = "ok", None, -1
        try:
            sub, sinfo = api.get_json(f"/ontologies/{acr}/latest_submission",
                                      {"display": "all", "display_context": "false", "display_links": "false"})
            if not sub or not isinstance(sub, dict) or sub.get("submissionId") is None:
                status = "no_submission"
                error = f"latest_submission http={sinfo.get('status')} {str(sinfo.get('body',''))[:200]}"
                sub = sub if isinstance(sub, dict) else None
            else:
                sub_id = int(sub["submissionId"])
                core.dump_json(sub, str(raw_dir / "latest_submission.json"))
            metrics = None
            if status == "ok":
                metrics, _ = api.get_json(f"/ontologies/{acr}/metrics", {"display_context": "false", "display_links": "false"})
                if metrics is not None:
                    core.dump_json(metrics, str(raw_dir / "metrics.json"))
        except AuthError:
            raise
        except Exception as exc:  # noqa: BLE001
            status, error, sub, metrics = "error", f"{type(exc).__name__}: {exc}", None, None
        ctx.state.upsert("catalog", {
            "acronym": acr, "name": onto.get("name"), "submission_id": sub_id, "status": status,
            "error": error, "fetched_at": now(), "api_seconds": round(time.perf_counter() - t0, 3),
            "ontology_json": json.dumps(onto), "submission_json": json.dumps(sub) if sub else None,
            "metrics_json": json.dumps(metrics) if metrics else None}, ("acronym",))
        counts[status] = counts.get(status, 0) + 1
        if i % 50 == 0 or i == len(todo):
            print(f"  catalog {i}/{len(todo)}  {counts}  ({time.time() - t_start:.0f}s)")
    ctx.event("catalog", None, "INFO", f"catalog phase finished: {counts}")
    return counts


def catalog_frame(ctx: Ctx):
    """Catalog as a DataFrame with the useful BioPortal fields pulled out."""
    import pandas as pd
    rows = []
    for r in ctx.state.q("SELECT * FROM catalog"):
        o = json.loads(r["ontology_json"] or "{}")
        s = json.loads(r["submission_json"] or "null") or {}
        m = json.loads(r["metrics_json"] or "null") or {}
        contacts = s.get("contact") or []
        rows.append({
            "acronym": r["acronym"], "name": r["name"], "submission_id": r["submission_id"],
            "catalog_status": r["status"], "catalog_error": r["error"], "catalog_fetched_at": r["fetched_at"],
            "bp_viewing_restriction": o.get("viewingRestriction"),
            "bp_summary_only": o.get("summaryOnly"),
            "bp_flat": o.get("flat"),
            "bp_is_view": bool(o.get("viewOf")),
            "bp_view_of": _tail(o.get("viewOf")) if o.get("viewOf") else None,
            "bp_categories": ";".join(sorted({_tail(x) for x in (o.get("hasDomain") or [])})),
            "bp_groups": ";".join(sorted({_tail(x) for x in (o.get("group") or [])})),
            "bp_ontology_type": _tail(o.get("ontologyType")) if o.get("ontologyType") else None,
            "bp_language": s.get("hasOntologyLanguage"),
            "bp_version": s.get("version"),
            "bp_released": s.get("released"),
            "bp_creation_date": s.get("creationDate"),
            "bp_modification_date": s.get("modificationDate"),
            "bp_submission_status": ";".join(_tail(x) for x in (s.get("submissionStatus") or [])),
            "bp_status": s.get("status"),
            "bp_description": (s.get("description") or "")[:5000],
            "bp_homepage": s.get("homepage"),
            "bp_documentation": s.get("documentation"),
            "bp_publication": s.get("publication") if isinstance(s.get("publication"), str) else json.dumps(s.get("publication")),
            "bp_ontology_uri": s.get("URI"),
            "bp_natural_language": ";".join(map(str, s.get("naturalLanguage") or [])) if isinstance(s.get("naturalLanguage"), list) else s.get("naturalLanguage"),
            "bp_master_file_name": s.get("masterFileName"),
            "bp_pull_location": s.get("pullLocation"),
            "bp_contact_names": ";".join(str(c.get("name")) for c in contacts if isinstance(c, dict)),
            "bp_classes": m.get("classes"), "bp_individuals": m.get("individuals"),
            "bp_properties": m.get("properties"), "bp_max_depth": m.get("maxDepth"),
            "bp_max_child_count": m.get("maxChildCount"), "bp_average_child_count": m.get("averageChildCount"),
            "bp_classes_with_one_child": m.get("classesWithOneChild"),
            "bp_classes_with_more_than_25_children": m.get("classesWithMoreThan25Children"),
            "bp_classes_with_no_definition": m.get("classesWithNoDefinition"),
            "bp_url": f"{ctx.cfg['bioportal_ui']}/{r['acronym']}",
        })
    return pd.DataFrame(rows)


def select_pilot(ctx: Ctx, n: int = 20, seed: int = 7) -> list:
    """Pick ontologies spread over size bins (BioPortal class counts) for a benchmark run."""
    df = catalog_frame(ctx)
    df = df[(df.catalog_status == "ok") & (df.bp_viewing_restriction != "private")
            & (df.bp_summary_only != True) & df.bp_classes.notna()]  # noqa: E712
    bins = [(0, 1_000), (1_000, 20_000), (20_000, 200_000), (200_000, 10**12)]
    rng = random.Random(seed)
    per_bin = max(1, n // len(bins))
    chosen: list = []
    for lo, hi in bins:
        pool = sorted(df[(df.bp_classes >= lo) & (df.bp_classes < hi)].acronym.tolist())
        rng.shuffle(pool)
        chosen += pool[:per_bin]
    rest = sorted(set(df.acronym) - set(chosen))
    rng.shuffle(rest)
    chosen += rest[: max(0, n - len(chosen))]
    return sorted(chosen)


# =============================================================================
# Phase B: downloads
# =============================================================================

def _stream_to_file(resp, dest: Path, max_bytes: int | None) -> tuple:
    digest = hashlib.sha256()
    size = 0
    part = dest.with_name(dest.name + ".part")
    with open(part, "wb") as fh:
        for chunk in resp.iter_content(chunk_size=1 << 20):
            if not chunk:
                continue
            fh.write(chunk)
            digest.update(chunk)
            size += len(chunk)
            if max_bytes and size > max_bytes:
                fh.close()
                part.unlink(missing_ok=True)
                return None, size
    os.replace(part, dest)
    return digest.hexdigest(), size


def _server_filename(resp) -> str | None:
    cd = resp.headers.get("Content-Disposition") or ""
    m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd)
    return m.group(1) if m else None


def _prepare_eval_file(raw_path: Path, master_name: str | None) -> dict:
    """Unpack zip/gzip if needed and pick the file to evaluate."""
    fmt = core.sniff_format(str(raw_path))
    out = {"eval_path": str(raw_path), "archive_type": None, "archive_members": None, "sniffed_format": fmt}
    if fmt == "zip":
        target = raw_path.parent / "extracted"
        target.mkdir(exist_ok=True)
        with zipfile.ZipFile(raw_path) as zf:
            members = [m for m in zf.infolist() if not m.is_dir()]
            names = [m.filename for m in members]
            if "[Content_Types].xml" in names:
                out.update(archive_type="office_document", archive_members=json.dumps(names[:500]),
                           sniffed_format="office_document")
                return out
            zf.extractall(target)
        pick = None
        if master_name:
            pick = next((m for m in members if Path(m.filename).name == master_name), None)
        if pick is None:
            ok_ext = (".owl", ".rdf", ".ttl", ".xml", ".nt", ".n3", ".jsonld", ".trig", ".obo")
            cands = [m for m in members if m.filename.lower().endswith(ok_ext)] or members
            pick = max(cands, key=lambda m: m.file_size)
        eval_path = target / pick.filename
        out.update(eval_path=str(eval_path), archive_type="zip", archive_members=json.dumps(names[:500]),
                   sniffed_format=core.sniff_format(str(eval_path)))
    elif fmt == "gzip":
        eval_path = raw_path.with_suffix(raw_path.suffix + ".unzipped")
        with gzip.open(raw_path, "rb") as src, open(eval_path, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 20)
        out.update(eval_path=str(eval_path), archive_type="gzip",
                   sniffed_format=core.sniff_format(str(eval_path)))
    return out


def _fetch(ctx: Ctx, acr: str, sub: int, variant: str, url: str, params: dict, dest_dir: Path,
           master_name: str | None, *, external: bool = False) -> dict:
    """Download one candidate file, retrying broken streams.

    ``url`` is a BioPortal API path, or a full URL when ``external`` is True.
    External URLs are fetched without the BioPortal API key.
    Returns {"outcome": ok|forbidden|too_large|http_error|html|stream_error, "result": row, "entry": log}.
    """
    max_bytes = int(ctx.cfg["max_download_gb"] * 2**30) if ctx.cfg["max_download_gb"] else None
    retries = max(1, int(ctx.cfg.get("download_stream_retries", 3)))
    read_timeout = ctx.cfg["download_read_timeout_s"]
    shown_url = url if external else f"{ctx.cfg['bioportal_api'].rstrip('/')}{url}"
    entry: dict = {"variant": variant, "url": shown_url, "params": params, "tries": []}
    last_error = None
    for attempt in range(1, retries + 1):
        if external:
            try:
                resp = requests.get(url, params=params, stream=True, timeout=(30, read_timeout),
                                    headers={"User-Agent": f"{NOTEBOOK_VERSION} (WiseOwl batch evaluation)"})
                info = {"status": resp.status_code}
            except requests.RequestException as exc:
                resp, info = None, {"error": f"{type(exc).__name__}: {str(exc)[:200]}"}
        else:
            resp, info = ctx.api.request(url, params, stream=True, read_timeout=read_timeout)
        status_code = info.get("status")
        if resp is None or resp.status_code != 200:
            body = None
            if resp is not None:
                try:
                    body = resp.text[:500]
                finally:
                    resp.close()
            entry["tries"].append({"status": status_code, "error": info.get("error"), "body": body})
            entry.update(status=status_code, body=body)
            if status_code in (401, 403):
                return {"outcome": "forbidden", "entry": entry,
                        "result": {"status": "not_downloadable", "http_status": status_code, "error": body}}
            if resp is not None:
                return {"outcome": "http_error", "entry": entry,
                        "result": {"status": "not_found" if status_code == 404 else "failed",
                                   "http_status": status_code, "error": body}}
            last_error = info.get("error")
            time.sleep(5 * attempt)
            continue
        ctype = resp.headers.get("Content-Type")
        clen = resp.headers.get("Content-Length")
        if max_bytes and clen and int(clen) > max_bytes:
            resp.close()
            entry["skipped"] = f"content-length {clen} over limit"
            return {"outcome": "too_large", "entry": entry,
                    "result": {"status": "skipped_too_large", "http_status": 200, "bytes": int(clen)}}
        server_name = _server_filename(resp)
        if not server_name and external:
            server_name = Path(requests.utils.urlparse(resp.url).path).name or None
        ext = Path(server_name).suffix if server_name else ".download"
        tag = safe_name(variant.replace(":", "-"))
        raw_path = dest_dir / f"{safe_name(acr)}_{sub}_{tag}{ext or '.download'}"
        try:
            sha, size = _stream_to_file(resp, raw_path, max_bytes)
        except Exception as exc:  # noqa: BLE001
            last_error = f"stream failed: {type(exc).__name__}: {str(exc)[:300]}"
            entry["tries"].append({"status": 200, "error": last_error})
            if attempt < retries:
                time.sleep(10 * attempt)
                continue
            return {"outcome": "stream_error", "entry": entry,
                    "result": {"status": "failed", "http_status": 200, "error": last_error}}
        finally:
            resp.close()
        if sha is None:
            entry["skipped"] = f"stream exceeded limit at {size} bytes"
            return {"outcome": "too_large", "entry": entry,
                    "result": {"status": "skipped_too_large", "http_status": 200, "bytes": size}}
        prep = _prepare_eval_file(raw_path, master_name)
        entry.update(status=200, bytes=size, sniffed=prep["sniffed_format"], tries_used=attempt)
        if prep["sniffed_format"] == "html":
            return {"outcome": "html", "entry": entry,
                    "result": {"status": "html_response", "http_status": 200, "error": "server returned HTML"}}
        return {"outcome": "ok", "entry": entry,
                "result": {"status": "ok", "variant": variant, "http_status": 200, "url": shown_url,
                           "raw_path": str(raw_path), "bytes": size, "sha256": sha, "content_type": ctype,
                           "server_filename": server_name, **prep,
                           "eval_bytes": Path(prep["eval_path"]).stat().st_size}}
    return {"outcome": "network_error", "entry": entry,
            "result": {"status": "failed", "error": last_error}}


def download_phase(ctx: Ctx, only: list | None = None, retry_failed: bool = False,
                   limit: int | None = None) -> dict:
    """Download each ontology's latest submission (RDF first, original as fallback)."""
    api = ctx.api
    df = catalog_frame(ctx)
    if df.empty:
        print("Catalog is empty: run the catalog phase first.")
        return {}
    df = df[df.catalog_status == "ok"]
    if only:
        wanted = {a.upper() for a in only}
        df = df[df.acronym.str.upper().isin(wanted)]
    df = df.assign(_size=df.bp_classes.fillna(10**12)).sort_values("_size")
    existing = {(r["acronym"], r["submission_id"]): r for r in ctx.state.q("SELECT * FROM downloads")}
    max_bytes = int(ctx.cfg["max_download_gb"] * 2**30) if ctx.cfg["max_download_gb"] else None
    counts: dict = {}
    rows = df.to_dict("records")
    if limit:
        rows = rows[:limit]
    _keep_awake(ctx, True)
    t_start = time.time()
    try:
        for i, row in enumerate(rows, 1):
            acr, sub = row["acronym"], int(row["submission_id"])
            prev = existing.get((acr, sub))
            if prev:
                ok_on_disk = prev["status"] == "ok" and prev["eval_path"] and Path(prev["eval_path"]).exists()
                if ok_on_disk or (prev["status"] != "ok" and not retry_failed):
                    counts["already_" + prev["status"]] = counts.get("already_" + prev["status"], 0) + 1
                    continue
            base = {"acronym": acr, "submission_id": sub, "run_id": ctx.run_id, "started_at": now(),
                    "attempts": (prev["attempts"] if prev else 0) + 1}
            if row["bp_viewing_restriction"] == "private":
                ctx.state.upsert("downloads", {**base, "status": "private", "finished_at": now()},
                                 ("acronym", "submission_id"))
                counts["private"] = counts.get("private", 0) + 1
                continue
            if str(row["bp_summary_only"]).lower() == "true":
                ctx.state.upsert("downloads", {**base, "status": "summary_only", "finished_at": now()},
                                 ("acronym", "submission_id"))
                counts["summary_only"] = counts.get("summary_only", 0) + 1
                continue
            dest_dir = ctx.paths["downloads"] / safe_name(acr) / str(sub)
            dest_dir.mkdir(parents=True, exist_ok=True)
            variants = []
            fmt = ctx.cfg["download_format"]
            if fmt:
                variants.append((f"{fmt}:submission", f"/ontologies/{acr}/submissions/{sub}/download", {"download_format": fmt}))
                variants.append((f"{fmt}:latest", f"/ontologies/{acr}/download", {"download_format": fmt}))
            if ctx.cfg["download_fallback_original"] or not fmt:
                variants.append(("original:submission", f"/ontologies/{acr}/submissions/{sub}/download", {}))
            attempts_log = []
            result = None
            t0 = time.perf_counter()
            for variant, path, params in variants:
                out = _fetch(ctx, acr, sub, variant, path, params, dest_dir, row.get("bp_master_file_name"))
                attempts_log.append(out["entry"])
                result = out["result"]
                if out["outcome"] in ("ok", "forbidden", "too_large"):
                    break
            if result is None:
                result = {"status": "failed", "error": "no download variants configured"}
            ctx.state.upsert("downloads", {**base, **result, "finished_at": now(),
                                           "seconds": round(time.perf_counter() - t0, 3),
                                           "attempts_log": json.dumps(attempts_log)},
                             ("acronym", "submission_id"))
            counts[result["status"]] = counts.get(result["status"], 0) + 1
            if result["status"] != "ok":
                ctx.event("download", acr, "WARNING", f"{result['status']}: {str(result.get('error'))[:300]}")
            if i % 25 == 0 or i == len(rows):
                print(f"  downloads {i}/{len(rows)}  {counts}  ({time.time() - t_start:.0f}s)")
    finally:
        _keep_awake(ctx, False)
    ctx.event("download", None, "INFO", f"download phase finished: {counts}")
    return counts


def add_local_file(ctx: Ctx, path: str, acronym: str, submission_id: int = 0, name: str | None = None) -> None:
    """Register a local ontology file so it goes through the same evaluation and export."""
    p = Path(path).resolve()
    prep = _prepare_eval_file(p, None)
    ctx.state.upsert("catalog", {"acronym": acronym, "name": name or acronym, "submission_id": submission_id,
                                 "status": "ok", "error": None, "fetched_at": now(), "api_seconds": 0,
                                 "ontology_json": json.dumps({"acronym": acronym, "name": name or acronym, "local_file": True}),
                                 "submission_json": None, "metrics_json": None}, ("acronym",))
    ctx.state.upsert("downloads", {"acronym": acronym, "submission_id": submission_id, "status": "ok",
                                   "variant": "local", "raw_path": str(p), "bytes": p.stat().st_size,
                                   "sha256": core.file_sha256(str(p), length=None), **prep,
                                   "eval_bytes": Path(prep["eval_path"]).stat().st_size,
                                   "started_at": now(), "finished_at": now(), "run_id": ctx.run_id,
                                   "attempts": 1}, ("acronym", "submission_id"))


# =============================================================================
# Phase C: evaluation with a restartable worker process
# =============================================================================

def _keep_awake(ctx: Ctx, on: bool) -> None:
    if not ctx.cfg.get("keep_awake") or os.name != "nt":
        return
    try:
        import ctypes
        es_continuous, es_system_required = 0x80000000, 0x00000001
        ctypes.windll.kernel32.SetThreadExecutionState(es_continuous | (es_system_required if on else 0))
    except Exception:  # noqa: BLE001
        pass


def _read_stage(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:  # noqa: BLE001
        return {}


class EvalWorker:
    """One background process holding BERT. Killed and restarted when needed."""

    def __init__(self, ctx: Ctx) -> None:
        self.ctx = ctx
        self.proc = None
        self.tq = None
        self.rq = None
        self.info: dict | None = None
        self.restarts = 0
        self.stage_file = ctx.paths["logs"] / "current_stage.json"
        cfg = ctx.cfg
        self.wcfg = {k: cfg[k] for k in (
            "device", "bert_name", "bert_max_length", "batch_size_gpu", "batch_size_cpu",
            "mixed_precision", "cpu_threads", "define_vectors_on_gpu_max_gb", "define_cpu_fallback",
            "define_min_tokens", "depth_target", "branch_target", "suggestion_threshold",
            "save_entity_tables", "entity_rows_max", "use_owlready2")}
        self.wcfg.update(log_dir=str(ctx.paths["logs"]), stage_file=str(self.stage_file))

    def alive(self) -> bool:
        return self.proc is not None and self.proc.is_alive()

    def start(self) -> dict:
        import multiprocessing as mp
        import owl4_worker
        code_dir = str(Path(core.__file__).resolve().parent)
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        # A fixed hash seed makes set order, and so WiseOwl's Flat on ontologies with
        # subclass cycles, the same on every run. The worker is a new Python process,
        # so it picks this up at start-up.
        os.environ["PYTHONHASHSEED"] = str(self.ctx.cfg.get("python_hash_seed", 0))
        mpc = mp.get_context("spawn")
        self.tq, self.rq = mpc.Queue(), mpc.Queue()
        self.proc = mpc.Process(target=owl4_worker.worker_main, args=(self.tq, self.rq, self.wcfg),
                                daemon=True, name="owl4-worker")
        self.proc.start()
        deadline = time.time() + self.ctx.cfg["worker_ready_timeout_s"]
        while True:
            try:
                msg = self.rq.get(timeout=2)
            except queue.Empty:
                if not self.proc.is_alive():
                    raise RuntimeError(f"worker exited while starting (exit code {self.proc.exitcode}); see logs/worker.log")
                if time.time() > deadline:
                    self.kill()
                    raise RuntimeError("worker did not become ready in time")
                continue
            if msg.get("type") == "ready":
                self.info = msg
                self.ctx.event("evaluate", None, "INFO", f"worker ready: {json.dumps(msg)}")
                return msg
            if msg.get("type") == "fatal":
                self.kill()
                raise RuntimeError(f"worker failed to start: {msg.get('error')}\n{msg.get('traceback')}")

    def kill(self) -> None:
        if self.proc is not None:
            try:
                self.proc.kill()
                self.proc.join(10)
            except Exception:  # noqa: BLE001
                pass
        for qobj in (self.tq, self.rq):
            try:
                qobj.cancel_join_thread()
                qobj.close()
            except Exception:  # noqa: BLE001
                pass
        self.proc = self.tq = self.rq = None

    def stop(self) -> None:
        if self.alive():
            try:
                self.tq.put(None)
                self.proc.join(30)
            except Exception:  # noqa: BLE001
                pass
        self.kill()

    def run(self, task: dict, timeout_s: float, ram_limit_bytes: int | None) -> dict:
        import psutil
        if not self.alive():
            if self.proc is not None:
                self.restarts += 1
            self.start()
        self.tq.put(task)
        t0 = time.time()
        peak = 0
        last_hb = t0
        pid = self.proc.pid
        while True:
            try:
                msg = self.rq.get(timeout=1.0)
                msg["parent_peak_rss_mb"] = round(peak / 2**20, 1)
                msg["wall_seconds"] = round(time.time() - t0, 3)
                return msg
            except queue.Empty:
                pass
            elapsed = time.time() - t0
            if not self.proc.is_alive():
                code = self.proc.exitcode
                stage = _read_stage(self.stage_file)
                self.kill()
                return {"status": "crashed", "error_type": "WorkerExit",
                        "error": f"worker process exited with code {code}", "stage": stage.get("stage"),
                        "parent_peak_rss_mb": round(peak / 2**20, 1), "wall_seconds": round(elapsed, 3)}
            try:
                rss = psutil.Process(pid).memory_info().rss
            except Exception:  # noqa: BLE001
                rss = 0
            peak = max(peak, rss)
            reason = None
            if ram_limit_bytes and rss > ram_limit_bytes:
                reason = ("memory_limit", f"worker RAM {rss / 2**30:.1f} GB over limit {ram_limit_bytes / 2**30:.1f} GB")
            elif elapsed > timeout_s:
                reason = ("timeout", f"no result after {elapsed / 60:.1f} min (limit {timeout_s / 60:.1f} min)")
            if reason:
                stage = _read_stage(self.stage_file)
                self.kill()
                return {"status": reason[0], "error_type": reason[0], "error": reason[1],
                        "stage": stage.get("stage"), "parent_peak_rss_mb": round(peak / 2**20, 1),
                        "wall_seconds": round(elapsed, 3)}
            if time.time() - last_hb >= self.ctx.cfg["heartbeat_s"]:
                stage = _read_stage(self.stage_file).get("stage")
                print(f"      ... {task['key']} running {elapsed / 60:.1f} min, stage={stage}, worker RAM {rss / 2**30:.1f} GB")
                last_hb = time.time()


def _mark_interrupted(ctx: Ctx) -> None:
    for r in ctx.state.q("SELECT acronym, submission_id, attempts FROM evaluations WHERE status='running'"):
        new = "gave_up" if r["attempts"] >= ctx.cfg["max_attempts"] else "interrupted"
        ctx.state.exec("UPDATE evaluations SET status=?, error=COALESCE(error,'') || ' [session ended while running]' "
                       "WHERE acronym=? AND submission_id=?", (new, r["acronym"], r["submission_id"]))
        ctx.event("evaluate", r["acronym"], "WARNING", f"found unfinished run from an earlier session -> {new}")


def evaluate_phase(ctx: Ctx, only: list | None = None, limit: int | None = None,
                   timeout_min: float | None = None) -> dict:
    """Evaluate every downloaded ontology that has no final result yet. Safe to re-run."""
    import psutil
    _mark_interrupted(ctx)
    rows = ctx.state.q(
        "SELECT d.acronym, d.submission_id, d.eval_path, d.eval_bytes, d.sniffed_format, "
        "e.status AS eval_status, e.attempts AS eval_attempts "
        "FROM downloads d LEFT JOIN evaluations e ON d.acronym=e.acronym AND d.submission_id=e.submission_id "
        "WHERE d.status='ok' ORDER BY d.eval_bytes ASC")
    if only:
        wanted = {a.upper() for a in only}
        rows = [r for r in rows if r["acronym"].upper() in wanted]
    todo = []
    for r in rows:
        st, att = r["eval_status"], r["eval_attempts"] or 0
        if st in TERMINAL_EVAL:
            continue
        if st in RETRYABLE_EVAL and att >= ctx.cfg["max_attempts"]:
            ctx.state.exec("UPDATE evaluations SET status='gave_up' WHERE acronym=? AND submission_id=?",
                           (r["acronym"], r["submission_id"]))
            continue
        todo.append(r)
    if limit:
        todo = todo[:limit]
    total_bytes = sum(r["eval_bytes"] or 0 for r in todo)
    print(f"{len(todo)} ontologies to evaluate ({total_bytes / 2**30:.2f} GB of files). "
          f"Already final: {len(rows) - len(todo)}.")
    if not todo:
        return {}
    ram_gb = ctx.cfg["worker_ram_limit_gb"] or round(psutil.virtual_memory().total * 0.85 / 2**30, 1)
    ram_limit = int(ram_gb * 2**30)
    tmin = timeout_min or ctx.cfg["eval_timeout_min"]
    print(f"Limits per ontology: {tmin:.0f} min, {ram_gb} GB worker RAM. Interrupt the kernel to pause; re-run to resume.")
    worker = EvalWorker(ctx)
    extra_props = _bp_definition_map(ctx)
    counts: dict = {}
    t_start = time.time()
    done_bytes = 0
    _keep_awake(ctx, True)
    current = None
    try:
        info = worker.start()
        print(f"Worker ready on {info['device']} ({info.get('gpu_name')}), model load {info['model_load_seconds']} s")
        for i, r in enumerate(todo, 1):
            acr, sub = r["acronym"], r["submission_id"]
            key = f"{acr}@{sub}"
            current = r
            attempts = (r["eval_attempts"] or 0) + 1
            ctx.state.upsert("evaluations", {"acronym": acr, "submission_id": sub, "status": "running",
                                             "attempts": attempts, "started_at": now(), "run_id": ctx.run_id,
                                             "timeout_min": tmin, "error": None, "error_type": None,
                                             "traceback": None, "stage_at_failure": None},
                             ("acronym", "submission_id"))
            out_dir = ctx.paths["results"] / safe_name(acr) / str(sub)
            task = {"key": key, "acronym": acr, "submission_id": sub, "path": r["eval_path"],
                    "out_dir": str(out_dir), "extra_definition_props": extra_props.get(acr, [])}
            msg = worker.run(task, tmin * 60, ram_limit)
            status = msg.get("status", "error")
            upd = {"acronym": acr, "submission_id": sub, "status": status, "finished_at": now(),
                   "seconds": msg.get("wall_seconds"), "parent_peak_rss_mb": msg.get("parent_peak_rss_mb"),
                   "error_type": msg.get("error_type"), "error": msg.get("error"),
                   "traceback": msg.get("traceback"), "stage_at_failure": msg.get("stage")}
            if status == "done":
                s = msg["summary"]
                s["wall_seconds"] = msg.get("wall_seconds")
                s["parent_peak_rss_mb"] = msg.get("parent_peak_rss_mb")
                upd.update(describe_score=s["describe_score"], define_score=s["define_score"],
                           connection_score=s["connection_score"], flat_score=s["flat_score"],
                           core_average=s["core_average"], summary_json=json.dumps(core.to_jsonable(s)),
                           result_path=msg.get("result_path"), entities_path=msg.get("entities_path"))
            else:
                # a failed re-run must not leave the previous run's scores behind
                upd.update(describe_score=None, define_score=None, connection_score=None, flat_score=None,
                           core_average=None, summary_json=None, result_path=None, entities_path=None)
                if status in RETRYABLE_EVAL and attempts >= ctx.cfg["max_attempts"]:
                    upd["status"] = "gave_up"
            ctx.state.upsert("evaluations", upd, ("acronym", "submission_id"))
            if status == "done" and not ctx.cfg.get("keep_downloads", True):
                shutil.rmtree(ctx.paths["downloads"] / safe_name(acr) / str(sub), ignore_errors=True)
            counts[upd["status"]] = counts.get(upd["status"], 0) + 1
            done_bytes += r["eval_bytes"] or 0
            elapsed = time.time() - t_start
            if status == "done":
                line = (f"avg {s['core_average']:.2f} (Describe {s['describe_score']}, Define {s['define_score']}, "
                        f"Connection {s['connection_score']}, Flat {s['flat_score']})")
                if s.get("define_bp_source") not in (None, "same_as_strict"):
                    line += f" | Define_bp {s['define_bp_score']}, avg_bp {s['core_average_bp']:.2f}"
                if s.get("entities") == 0:
                    line += " | EMPTY (0 entities)"
            else:
                line = f"{upd['status'].upper()}: {str(upd['error'])[:160]}"
                ctx.event("evaluate", acr, "WARNING", f"{upd['status']} at stage {upd['stage_at_failure']}: {upd['error']}")
            print(f"[{i}/{len(todo)}] {acr:<18} {msg.get('wall_seconds', 0):>8.1f}s  {line}")
            if i % 25 == 0:
                rate = done_bytes / max(elapsed, 1)
                left = (total_bytes - done_bytes) / rate if rate > 0 else float("nan")
                print(f"    progress: {counts}; elapsed {elapsed / 3600:.2f} h; rough time left by bytes "
                      f"{left / 3600:.1f} h (large files are slower per byte, so treat as a lower bound)")
        current = None
    except KeyboardInterrupt:
        print("\nPaused. The ontology that was running goes back to the queue. Re-run this cell to continue.")
        if current is not None:
            ctx.state.exec("UPDATE evaluations SET status='interrupted', attempts=MAX(attempts-1,0), "
                           "error='paused by user' WHERE acronym=? AND submission_id=?",
                           (current["acronym"], current["submission_id"]))
    finally:
        worker.stop()
        _keep_awake(ctx, False)
    ctx.event("evaluate", None, "INFO", f"evaluate phase finished: {counts}; worker restarts {worker.restarts}")
    ctx.state.exec("UPDATE runs SET finished_at=?, phase=? WHERE run_id=?", (now(), "evaluate", ctx.run_id))
    return counts


# =============================================================================
# Version 1.1: BioPortal-aware Define and repairs for an existing work folder
# =============================================================================

WISEOWL_DEFINE_IRIS = {str(p) for p in core.DEFINE_DEFINITION_PROPS}


def _bp_definition_iris(sub: dict | None) -> list:
    """Definition property IRIs that BioPortal records for a submission."""
    if not isinstance(sub, dict):
        return []
    value = sub.get("definitionProperty")
    if not value:
        return []
    items = value if isinstance(value, list) else [value]
    out = []
    for x in items:
        if isinstance(x, dict):
            x = x.get("@id")
        if isinstance(x, str) and x.startswith("http"):
            out.append(x)
    return out


def _bp_definition_map(ctx: Ctx) -> dict:
    out = {}
    for r in ctx.state.q("SELECT acronym, submission_json FROM catalog"):
        try:
            out[r["acronym"]] = _bp_definition_iris(json.loads(r["submission_json"] or "null"))
        except ValueError:
            out[r["acronym"]] = []
    return out


def plan_v11(ctx: Ctx) -> dict:
    """Work out which ontologies the 1.1 update needs to touch. Changes nothing."""
    cat = {r["acronym"]: r for r in ctx.state.q("SELECT acronym, submission_json, metrics_json FROM catalog")}
    downloads = {r["acronym"]: r for r in ctx.state.q("SELECT acronym, status, variant FROM downloads")}
    evals = {r["acronym"]: r for r in ctx.state.q("SELECT acronym, status, summary_json FROM evaluations")}
    plan: dict = {"download_failed": [], "parse_failed": [], "empty": [], "define_candidates": [],
                  "define_reasons": {}, "flat_recheck": [], "already_v11": []}
    for acr, d in downloads.items():
        if d["status"] in ("failed", "html_response", "not_found"):
            plan["download_failed"].append(acr)
    for acr, e in evals.items():
        if e["status"] == "parse_failed":
            plan["parse_failed"].append(acr)
            continue
        if e["status"] != "done":
            continue
        summ = json.loads(e["summary_json"] or "{}")
        if summ.get("entities") == 0:
            plan["empty"].append(acr)
            continue
        if "define_bp_score" in summ:
            plan["already_v11"].append(acr)
            continue
        if (summ.get("flat_cycle_back_edges") or 0) > 0 and (summ.get("flat_max_depth") or 0) <= 6:
            plan["flat_recheck"].append(acr)
        c = cat.get(acr) or {}
        sub = json.loads(c.get("submission_json") or "null")
        met = json.loads(c.get("metrics_json") or "null") or {}
        reasons = []
        extra = [i for i in _bp_definition_iris(sub) if i not in WISEOWL_DEFINE_IRIS]
        if extra:
            reasons.append("BioPortal definition property " + ", ".join(extra))
        try:
            bp_defined = int(met.get("classes") or 0) - int(met.get("classesWithNoDefinition") or 0)
        except (TypeError, ValueError):
            bp_defined = 0
        wo_defined = summ.get("define_defined") or 0
        if bp_defined >= 10 and wo_defined < 0.5 * bp_defined:
            reasons.append(f"BioPortal counts {bp_defined} definitions, WiseOwl found {wo_defined}")
        if reasons:
            plan["define_candidates"].append(acr)
            plan["define_reasons"][acr] = "; ".join(reasons)
    for k in ("download_failed", "parse_failed", "empty", "define_candidates", "flat_recheck"):
        plan[k] = sorted(plan[k])
    return plan


def _alternate_sources(ctx: Ctx, acr: str, sub: int, current_variant: str | None) -> list:
    """Other places to get the same ontology: BioPortal's original upload, then an OWL file for OBO ontologies."""
    row = ctx.state.one("SELECT submission_json FROM catalog WHERE acronym=?", (acr,)) or {}
    subj = json.loads(row.get("submission_json") or "null") or {}
    alts = []
    if not (current_variant or "").startswith("original"):
        alts.append(("original:submission", f"/ontologies/{acr}/submissions/{sub}/download", {}, False))
    lang = str(subj.get("hasOntologyLanguage") or "").upper()
    dl = ctx.state.one("SELECT sniffed_format FROM downloads WHERE acronym=? AND submission_id=?", (acr, sub)) or {}
    if ctx.cfg.get("obo_owl_fallback", True) and (lang == "OBO" or dl.get("sniffed_format") == "obo"):
        cands = []
        pull = subj.get("pullLocation")
        if isinstance(pull, str) and pull.startswith("http"):
            if pull.lower().endswith(".obo"):
                cands.append(pull[:-4] + ".owl")
            elif pull.lower().endswith((".owl", ".rdf", ".ttl", ".owl.gz")):
                cands.append(pull)
        cands.append(f"http://purl.obolibrary.org/obo/{acr.lower()}.owl")
        for i, u in enumerate(dict.fromkeys(cands)):
            alts.append((f"obo_owl:{i}", u, {}, True))
    return alts


def fetch_from_url(ctx: Ctx, acronym: str, url: str, *, evaluate: bool = True, force: bool = False) -> dict:
    """Get an ontology file from another address (for example the OBO Foundry) and score it.

    Skipped when the ontology already has a non-empty score, unless force=True.
    The download_variant column records where the file came from.
    """
    d = ctx.state.one("SELECT * FROM downloads WHERE acronym=?", (acronym,))
    if d is None:
        print(f"{acronym}: not in the downloads table; run the download phase first.")
        return {"outcome": "unknown_acronym"}
    if not force and _is_good(ctx, acronym):
        print(f"{acronym}: already scored; nothing to do (use force=True to fetch anyway).")
        return {"outcome": "already_scored"}
    sub = int(d["submission_id"])
    dest = ctx.paths["downloads"] / safe_name(acronym) / str(sub)
    dest.mkdir(parents=True, exist_ok=True)
    host = requests.utils.urlparse(url).netloc or "url"
    print(f"{acronym}: downloading {url} ...")
    out = _fetch(ctx, acronym, sub, f"url:{host}", url, {}, dest, None, external=True)
    print(f"{acronym}: {out['outcome']}, {out['entry'].get('bytes')} bytes")
    if out["outcome"] != "ok":
        return out
    old_log = json.loads(d.get("attempts_log") or "[]")
    ctx.state.upsert("downloads", {"acronym": acronym, "submission_id": sub, **out["result"],
                                   "finished_at": now(), "run_id": ctx.run_id,
                                   "attempts_log": json.dumps(old_log + [out["entry"]])},
                     ("acronym", "submission_id"))
    if evaluate:
        requeue(ctx, statuses=("parse_failed", "done", "error", "crashed", "gave_up"), acronyms=[acronym])
        print(evaluate_phase(ctx, only=[acronym]))
    return out


def _eval_row(ctx: Ctx, acr: str) -> dict:
    return ctx.state.one("SELECT * FROM evaluations WHERE acronym=?", (acr,)) or {}


def _is_good(ctx: Ctx, acr: str) -> bool:
    e = _eval_row(ctx, acr)
    if e.get("status") != "done":
        return False
    return json.loads(e.get("summary_json") or "{}").get("entities", 0) > 0


def repair_phase(ctx: Ctx) -> dict:
    """Retry failed downloads, re-parse failures with the 1.1 parser, then try alternate sources."""
    plan = plan_v11(ctx)
    report: dict = {"retried_downloads": {}, "reparsed": {}, "alternates": {}}

    # 1. failed downloads (PR and similar)
    if plan["download_failed"]:
        print(f"== Retrying {len(plan['download_failed'])} failed downloads: {plan['download_failed']}")
        report["retried_downloads"] = download_phase(ctx, only=plan["download_failed"], retry_failed=True)

    # 2. re-check files and re-evaluate with the new parser
    targets = sorted(set(plan["parse_failed"]) | set(plan["empty"]) | set(plan["download_failed"]))
    for r in ctx.state.q("SELECT * FROM downloads WHERE status='ok'"):
        if r["acronym"] in targets and r["raw_path"] and Path(r["raw_path"]).exists():
            master = None
            sj = ctx.state.one("SELECT submission_json FROM catalog WHERE acronym=?", (r["acronym"],))
            if sj and sj["submission_json"]:
                master = (json.loads(sj["submission_json"]) or {}).get("masterFileName")
            prep = _prepare_eval_file(Path(r["raw_path"]), master)
            ctx.state.exec("UPDATE downloads SET eval_path=?, sniffed_format=?, archive_type=?, archive_members=? "
                           "WHERE acronym=? AND submission_id=?",
                           (prep["eval_path"], prep["sniffed_format"], prep["archive_type"],
                            prep["archive_members"], r["acronym"], r["submission_id"]))
    if targets:
        print(f"\n== Re-evaluating {len(targets)} ontologies with the 1.1 parser")
        requeue(ctx, statuses=("parse_failed", "done", "error", "crashed", "gave_up"), acronyms=targets)
        report["reparsed"] = evaluate_phase(ctx, only=targets)

    # 3. alternate sources for anything still failing or empty
    still = [a for a in targets
             if not _is_good(ctx, a)
             and (ctx.state.one("SELECT status FROM downloads WHERE acronym=?", (a,)) or {}).get("status") == "ok"]
    if still:
        print(f"\n== Trying alternate sources for {len(still)} ontologies: {still}")
    for acr in still:
        d = ctx.state.one("SELECT * FROM downloads WHERE acronym=?", (acr,))
        sub = int(d["submission_id"])
        e = _eval_row(ctx, acr)
        if (e.get("error") or "").startswith("not_an_ontology"):
            report["alternates"][acr] = "not an ontology; skipped"
            continue
        dest_dir = ctx.paths["downloads"] / safe_name(acr) / str(sub)
        dest_dir.mkdir(parents=True, exist_ok=True)
        tried = []
        for variant, url, params, external in _alternate_sources(ctx, acr, sub, d["variant"]):
            out = _fetch(ctx, acr, sub, variant, url, params, dest_dir, None, external=external)
            tried.append(f"{variant}: {out['outcome']}")
            if out["outcome"] != "ok":
                continue
            old_log = json.loads(d.get("attempts_log") or "[]")
            ctx.state.upsert("downloads", {"acronym": acr, "submission_id": sub, **out["result"],
                                           "finished_at": now(), "run_id": ctx.run_id,
                                           "attempts_log": json.dumps(old_log + [out["entry"]])},
                             ("acronym", "submission_id"))
            requeue(ctx, statuses=("parse_failed", "done", "error", "crashed", "gave_up"), acronyms=[acr])
            evaluate_phase(ctx, only=[acr])
            if _is_good(ctx, acr):
                tried.append("scored")
                break
        report["alternates"][acr] = tried
        print(f"   {acr}: {' -> '.join(tried) if tried else 'no alternate sources'}")
    ctx.event("repair", None, "INFO", json.dumps(report, default=str)[:3900])
    return report


def _older_bp_results(ctx: Ctx) -> list:
    """Ontologies whose BioPortal-aware Define used extra properties under an older core version."""
    out = []
    for r in ctx.state.q("SELECT acronym, summary_json FROM evaluations WHERE status='done'"):
        summ = json.loads(r["summary_json"] or "{}")
        src = summ.get("define_bp_source") or ""
        if "define_bp_score" in summ and src and not src.startswith("same_as_strict") \
                and summ.get("core4_version") != core.CORE4_VERSION:
            out.append(r["acronym"])
    return sorted(out)


def define_bp_phase(ctx: Ctx, full: bool = False) -> dict:
    """Re-score ontologies whose definitions live in a property WiseOwl does not read.

    Also re-scores ontologies whose BioPortal-aware Define was computed by an
    older core version (1.1.1 ignores rdfs:isDefinedBy and link values).
    full=True re-scores every scored ontology (about as long as the first run).
    """
    plan = plan_v11(ctx)
    older = _older_bp_results(ctx)
    for a in older:
        plan["define_candidates"].append(a)
        plan["define_reasons"][a] = "BioPortal-aware Define from an older version; re-score with 1.1.1 rules"
    if full:
        rows = ctx.state.q("SELECT acronym FROM evaluations WHERE status='done'")
        cands = sorted(r["acronym"] for r in rows)
    else:
        cands = sorted(set(plan["define_candidates"]) | set(plan["flat_recheck"]))
        for a in cands:
            reason = plan["define_reasons"].get(a, "")
            if a in plan["flat_recheck"]:
                reason = (reason + "; " if reason else "") + "subclass cycles and shallow taxonomy: re-check Flat with a fixed hash seed"
            print(f"  {a:<18} {reason}")
    if not cands:
        print("No ontologies need the BioPortal-aware Define.")
        return {}
    print(f"\n== Re-scoring {len(cands)} ontologies with version 1.1")
    requeue(ctx, statuses=("done",), acronyms=cands)
    return evaluate_phase(ctx, only=cands)


def requeue(ctx: Ctx, statuses: tuple = ("timeout", "memory_limit", "gave_up", "error", "crashed"),
            acronyms: list | None = None) -> int:
    """Put finished-but-failed evaluations back in the queue (attempts reset)."""
    sql = f"SELECT acronym, submission_id FROM evaluations WHERE status IN ({','.join('?' * len(statuses))})"
    rows = ctx.state.q(sql, tuple(statuses))
    if acronyms:
        wanted = {a.upper() for a in acronyms}
        rows = [r for r in rows if r["acronym"].upper() in wanted]
    for r in rows:
        ctx.state.exec("UPDATE evaluations SET status='pending', attempts=0 WHERE acronym=? AND submission_id=?",
                       (r["acronym"], r["submission_id"]))
    ctx.event("evaluate", None, "INFO", f"requeued {len(rows)} evaluations with status in {statuses}")
    return len(rows)


def status(ctx: Ctx) -> dict:
    out = {}
    for table, col in (("catalog", "status"), ("downloads", "status"), ("evaluations", "status")):
        out[table] = {r[col]: r["n"] for r in ctx.state.q(f"SELECT {col}, COUNT(*) AS n FROM {table} GROUP BY {col}")}
    t = ctx.state.one("SELECT SUM(seconds) AS s, COUNT(*) AS n FROM evaluations WHERE status='done'")
    out["evaluation_hours_done"] = round((t["s"] or 0) / 3600, 2)
    return out


# =============================================================================
# Parity check against WiseOwl's own test files
# =============================================================================

WISEOWL_RAW = "https://raw.githubusercontent.com/aryand1/WiseOwl/main/tests/data"


def parity_check(ctx: Ctx) -> dict:
    """Run tiny.owl and GoodRelations.owl through the worker and compare with WiseOwl's golden scores."""
    pdir = ctx.work / "parity"
    pdir.mkdir(exist_ok=True)
    for fname in ("tiny.owl", "GoodRelations.owl", "golden_tiny.json", "golden_goodrelations.json"):
        dest = pdir / fname
        if not dest.exists():
            r = requests.get(f"{WISEOWL_RAW}/{fname}", timeout=60)
            r.raise_for_status()
            dest.write_bytes(r.content)
    worker = EvalWorker(ctx)
    report: dict = {"time": now(), "cases": []}
    try:
        info = worker.start()
        report["worker"] = info
        for name, fname, golden in (("tiny", "tiny.owl", "golden_tiny.json"),
                                    ("goodrelations", "GoodRelations.owl", "golden_goodrelations.json")):
            expected = json.loads((pdir / golden).read_text())["scores"]
            msg = worker.run({"key": f"parity:{name}", "path": str(pdir / fname),
                              "out_dir": str(pdir / f"out_{name}")}, 600, None)
            if msg.get("status") != "done":
                report["cases"].append({"name": name, "ok": False, "error": msg.get("error"),
                                        "traceback": msg.get("traceback")})
                continue
            got = {k: msg["summary"][k] for k in core.CORE_LABELS}
            exp = {k: expected[k] for k in core.CORE_LABELS}
            report["cases"].append({"name": name, "ok": got == exp, "got": got, "expected": exp,
                                    "seconds": msg.get("wall_seconds")})
    finally:
        worker.stop()
    report["all_ok"] = all(c["ok"] for c in report["cases"])
    core.dump_json(report, str(ctx.paths["exports"] / "parity_check.json"))
    ctx.event("parity", None, "INFO" if report["all_ok"] else "ERROR", json.dumps(report["cases"]))
    return report


# =============================================================================
# Search latency benchmark (for the 2 second web target)
# =============================================================================

DEFAULT_KEYWORDS = [
    "diabetes", "cancer", "melanoma", "heart", "kidney", "liver", "asthma", "influenza",
    "gene", "protein", "cell", "neuron", "brain", "bone", "blood pressure", "vaccine",
    "antibiotic", "drug", "pregnancy", "obesity", "sleep", "anxiety", "depression",
    "mouse", "zebrafish", "plant", "soil", "climate", "food", "nutrition",
    "clinical trial", "imaging", "surgery", "pathology", "genome", "rna", "enzyme",
    "infection", "covid-19", "rare disease",
]


def search_benchmark(ctx: Ctx, keywords: list | None = None, repeats: int = 1) -> Any:
    """Time BioPortal /search and /recommender for each keyword."""
    import pandas as pd
    api = ctx.api
    rows = []
    for rep in range(repeats):
        for kw in keywords or DEFAULT_KEYWORDS:
            for endpoint, path, params in (
                    ("search", "/search", {"q": kw, "pagesize": 100, "display_context": "false",
                                           "display_links": "false", "include": "prefLabel"}),
                    ("recommender", "/recommender", {"input": kw, "input_type": 2, "output_type": 1,
                                                     "display_context": "false", "display_links": "false"})):
                data, info = api.get_json(path, params)
                count, onts, top = None, None, None
                try:
                    if endpoint == "search" and isinstance(data, dict):
                        coll = data.get("collection") or []
                        count = data.get("totalCount", len(coll))
                        acrs = [_tail((c.get("links") or {}).get("ontology") or "") for c in coll]
                        acrs = [a for a in acrs if a]
                        onts = len(set(acrs))
                        top = ";".join(pd.Series(acrs).value_counts().head(10).index) if acrs else ""
                    elif endpoint == "recommender" and isinstance(data, list):
                        count = len(data)
                        names = []
                        for rec in data:
                            for o in rec.get("ontologies") or []:
                                names.append(o.get("acronym") or _tail(o.get("@id", "")))
                        onts = len(set(names))
                        top = ";".join(names[:10])
                except Exception as exc:  # noqa: BLE001
                    info["error"] = f"parse: {exc}"
                row = {"ts": now(), "run_id": ctx.run_id, "endpoint": endpoint, "keyword": kw,
                       "http_status": info.get("status"), "seconds": info.get("seconds"),
                       "result_count": count, "distinct_ontologies": onts, "top_ontologies": top,
                       "error": info.get("error") or (info.get("body") if data is None else None)}
                ctx.state.conn.execute(
                    "INSERT INTO search_benchmark(ts, run_id, endpoint, keyword, http_status, seconds, "
                    "result_count, distinct_ontologies, top_ontologies, error) VALUES (?,?,?,?,?,?,?,?,?,?)",
                    tuple(row.values()))
                ctx.state.conn.commit()
                rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(ctx.paths["exports"] / f"search_benchmark_{ctx.run_id}.csv", index=False)
    return df


# =============================================================================
# Phase D: exports
# =============================================================================

COLUMN_DOCS = {
    "acronym": "BioPortal ontology acronym (primary key per provider).",
    "name": "Ontology name from BioPortal.",
    "submission_id": "BioPortal submission id that was downloaded and scored (-1 if none).",
    "catalog_status": "ok, no_submission, or error when fetching metadata.",
    "download_status": "ok, not_downloadable (license 401/403), private, summary_only, skipped_too_large, not_found, html_response, failed.",
    "download_variant": "Which download worked: rdf:submission, rdf:latest, original:submission, or local.",
    "download_bytes": "Size of the downloaded file in bytes.",
    "eval_bytes": "Size of the file that was parsed (after unzip).",
    "download_sha256": "SHA-256 of the downloaded file.",
    "eval_status": "done, parse_failed, timeout, memory_limit, memory_error, error, crashed, interrupted, gave_up, pending.",
    "eval_attempts": "How many times evaluation was started for this ontology.",
    "eval_seconds": "Wall time of the evaluation seen by the notebook, in seconds.",
    "web_status": "Status for the web page: scored, empty, not_an_ontology, not_downloadable, parse_failed, timed_out, too_large, failed, pending, no_submission.",
    "define_bp_score": "BioPortal-aware Define (0-10): same formula as Define, but also reads the ontology's own definition property (BioPortal definitionProperty, or a property named/labelled 'definition').",
    "define_bp_defined": "Entities with a definition under the BioPortal-aware property list.",
    "define_bp_source": "Extra definition properties used for define_bp_score, or same_as_strict when there were none.",
    "core_average_bp": "Mean of Describe, BioPortal-aware Define, Connection, Flat. The web page ranks by this.",
    "core_average_bp_2dp": "core_average_bp rounded to 2 decimals.",
    "empty_ontology": "True when the parsed file has no classes or individuals.",
    "describe_score": "WiseOwl Describe (0-10): 10 x share of entities with a descriptive annotation.",
    "define_score": "WiseOwl Define (0-10): 10 x mean of 0.4 x match + 0.6 x adequacy; undefined entities count as 0.",
    "connection_score": "WiseOwl Connection (0-10): 10 x (0.7 coverage + 0.2 diversity + 0.1 richness).",
    "flat_score": "WiseOwl Flat (integer 0-10): round((depth score + breadth score) / 2).",
    "core_average": "Mean of the four core scores, full precision (the WiseOwl average).",
    "core_average_2dp": "core_average rounded to 2 decimals, as the WiseOwl dashboard shows it.",
    "weakest_metric": "Core metric with the lowest score.",
    "suggestions": "WiseOwl suggestion sentences for metrics below the threshold.",
    "summary_banner": "WiseOwl summary sentence (core part).",
    "ontology_iri": "owl:Ontology IRI in the file, or a file hash if none is declared.",
    "version_iri": "owl:versionIRI, or a file hash if none is declared.",
    "sniffed_format": "Format detected from file content before parsing.",
    "parse_format": "rdflib format that parsed the file.",
    "triple_count": "Number of RDF triples in the parsed file (imports are not followed).",
    "entities": "Classes plus individuals: the population the metrics score.",
    "classes": "WiseOwl class set (typed classes, subClassOf participants, SKOS concepts).",
    "named_classes": "classes minus owl:Thing and owl:Nothing.",
    "individuals": "Subjects typed with a class that are not classes.",
    "deprecated_entities": "Entities marked owl:deprecated true (still counted by WiseOwl).",
    "owl_imports_count": "Number of owl:imports; imported ontologies are not loaded or scored.",
    "describe_described": "Entities counted as described.",
    "define_defined": "Entities with a non-empty definition.",
    "define_cosine_mean": "Mean BERT label/definition cosine (used for the z-score).",
    "define_cosine_std": "Std of the cosines (used for the z-score).",
    "define_definitions_truncated": "Definitions cut at the BERT max length (128 tokens).",
    "define_device": "Device used for Define (cuda or cpu; cpu after a GPU memory fallback).",
    "connection_coverage": "Share of entities with at least one object-property link.",
    "connection_diversity": "Mean of min(distinct properties / 5, 1).",
    "connection_richness": "Mean of min(log11(links + 1), 1).",
    "flat_max_depth": "Longest root-to-leaf path in the class taxonomy (in nodes).",
    "flat_avg_branching": "Mean children per parent in the class taxonomy.",
    "gpu_peak_allocated_mb": "Peak GPU memory allocated by PyTorch during this ontology.",
    "parent_peak_rss_mb": "Peak worker RAM seen by the notebook while this ontology ran.",
    "wiseowl_source_version": "WiseOwl version the metric code was copied from.",
    "core4_version": "Version of this four-metric copy.",
    "provider": "Ontology source (bioportal for now).",
    "catalog_error": "Error text when BioPortal metadata could not be fetched.",
    "catalog_fetched_at": "When the BioPortal metadata was fetched (UTC).",
    "parse_attempts": "How many rdflib formats were tried before one worked.",
    "subjects": "Distinct RDF subjects in the file.",
    "distinct_predicates": "Distinct RDF predicates in the file.",
    "annotation_properties": "Subjects typed owl:AnnotationProperty (they count as Describe signals).",
    "object_properties_declared": "Subjects typed owl:ObjectProperty.",
    "datatype_properties_declared": "Subjects typed owl:DatatypeProperty.",
    "owl_classes_typed": "Subjects typed owl:Class.",
    "rdfs_classes_typed": "Subjects typed rdfs:Class.",
    "skos_concepts": "Subjects typed skos:Concept (WiseOwl counts them as classes).",
    "owl_restrictions": "Subjects typed owl:Restriction.",
    "subclassof_triples": "rdfs:subClassOf triples.",
    "equivalentclass_triples": "owl:equivalentClass triples.",
    "entity_rows_written": "Rows written to entities.csv.gz for this ontology.",
    "rss_mb_now": "Worker RAM right after this ontology finished.",
    "process_lifetime_peak_mb": "Windows only: peak RAM of the worker process since it started.",
    "bert_name": "Hugging Face model used by Define.",
    "mixed_precision": "Whether fp16/bf16 autocast was on (off keeps WiseOwl parity).",
    "encoder_device": "Device the BERT encoder was loaded on.",
    "encoder_batch_size": "BERT batch size.",
    "torch_version": "PyTorch version in the worker.",
    "transformers_version": "transformers version in the worker.",
    "rdflib_version": "rdflib version in the worker.",
    "wall_seconds": "Evaluation wall time seen by the notebook.",
    "gpu_peak_reserved_mb": "Peak GPU memory reserved by PyTorch during this ontology.",
}
PREFIX_DOCS = {
    "bp_": "BioPortal metadata (ontology record, latest submission, or BioPortal metrics endpoint).",
    "time_": "Seconds spent in this stage of the evaluation.",
    "define_": "Define metric detail.",
    "connection_": "Connection metric detail.",
    "flat_": "Flat metric detail.",
    "describe_": "Describe metric detail.",
    "download_": "Download detail.",
    "eval_": "Evaluation bookkeeping (status, times, errors, file paths).",
}
WEB_STATUS = {
    "done": "scored", "parse_failed": "parse_failed", "timeout": "timed_out", "memory_limit": "too_large",
    "memory_error": "too_large", "gave_up": "failed", "error": "failed", "crashed": "failed",
}


def results_frame(ctx: Ctx):
    import pandas as pd
    cat = catalog_frame(ctx)
    dl = pd.DataFrame(ctx.state.q("SELECT * FROM downloads"))
    ev = pd.DataFrame(ctx.state.q("SELECT * FROM evaluations"))
    if not dl.empty:
        dl = dl.rename(columns={c: f"download_{c}" for c in dl.columns if c not in ("acronym", "submission_id")})
        dl = dl.rename(columns={"download_eval_bytes": "eval_bytes", "download_sha256": "download_sha256"})
        cat = cat.merge(dl, on=["acronym", "submission_id"], how="left")
    if not ev.empty:
        summaries = []
        for sj in ev["summary_json"].fillna("{}"):
            summaries.append(json.loads(sj))
        sdf = pd.DataFrame(summaries)
        drop = [c for c in ("describe_score", "define_score", "connection_score", "flat_score", "core_average") if c in sdf.columns]
        sdf = sdf.drop(columns=drop)
        ev = ev.drop(columns=["summary_json"])
        ev = ev.rename(columns={c: f"eval_{c}" for c in ev.columns
                                if c not in ("acronym", "submission_id", "describe_score", "define_score",
                                             "connection_score", "flat_score", "core_average")})
        ev = pd.concat([ev.reset_index(drop=True), sdf.reset_index(drop=True)], axis=1)
        cat = cat.merge(ev, on=["acronym", "submission_id"], how="left")
    if "core_average" in cat.columns:
        cat["core_average_2dp"] = cat["core_average"].round(2)
        if "define_bp_score" not in cat.columns:
            cat["define_bp_score"] = float("nan")
        if "define_bp_source" not in cat.columns:
            cat["define_bp_source"] = None
        scored = cat["core_average"].notna()
        missing = scored & cat["define_bp_score"].isna()
        cat.loc[missing, "define_bp_score"] = cat.loc[missing, "define_score"]
        cat.loc[missing, "define_bp_source"] = "same_as_strict (1.0 result, not re-checked)"
        cat["core_average_bp"] = (cat["describe_score"] + cat["define_bp_score"]
                                  + cat["connection_score"] + cat["flat_score"]) / 4.0
        cat["core_average_bp_2dp"] = cat["core_average_bp"].round(2)

    def web_status(row):
        if row.get("catalog_status") == "no_submission":
            return "no_submission"
        es = row.get("eval_status")
        if es == "done" and row.get("entities") == 0:
            return "empty"
        if es == "parse_failed" and str(row.get("eval_error") or "").startswith("not_an_ontology"):
            return "not_an_ontology"
        if isinstance(es, str) and es in WEB_STATUS:
            return WEB_STATUS[es]
        ds = row.get("download_status")
        if ds in ("not_downloadable", "private", "summary_only"):
            return "not_downloadable"
        if ds == "skipped_too_large":
            return "too_large"
        if isinstance(ds, str) and ds not in ("ok",):
            return "failed"
        return "pending"
    cat["web_status"] = cat.apply(web_status, axis=1)
    cat.insert(0, "provider", ctx.cfg["provider"])
    return cat


def _sql(v: Any) -> str:
    import math
    if v is None:
        return "NULL"
    if isinstance(v, float):
        return "NULL" if math.isnan(v) else repr(v)
    if isinstance(v, (int,)) and not isinstance(v, bool):
        return str(v)
    if isinstance(v, bool):
        return "1" if v else "0"
    return "'" + str(v).replace("'", "''") + "'"


D1_SCHEMA = """-- Cloudflare D1 / SQLite schema for the web search endpoint.
CREATE TABLE IF NOT EXISTS ontology_scores (
  provider TEXT NOT NULL,
  acronym TEXT NOT NULL,
  name TEXT,
  description TEXT,
  categories TEXT,
  ontology_language TEXT,
  submission_id INTEGER,
  version TEXT,
  released TEXT,
  bioportal_url TEXT,
  file_sha256 TEXT,
  status TEXT NOT NULL,
  status_detail TEXT,
  describe_score REAL,
  define_score REAL,
  define_bp_score REAL,
  define_bp_source TEXT,
  connection_score REAL,
  flat_score REAL,
  core_average REAL,
  core_average_bp REAL,
  logical_consistency_score REAL,
  structural_dist_score REAL,
  semantic_dist_score REAL,
  triple_count INTEGER,
  entity_count INTEGER,
  class_count INTEGER,
  wiseowl_version TEXT,
  core4_version TEXT,
  device TEXT,
  evaluated_at TEXT,
  updated_at TEXT,
  PRIMARY KEY (provider, acronym)
);
CREATE INDEX IF NOT EXISTS idx_scores_avg ON ontology_scores(core_average);
CREATE INDEX IF NOT EXISTS idx_scores_avg_bp ON ontology_scores(core_average_bp);
CREATE INDEX IF NOT EXISTS idx_scores_status ON ontology_scores(status);
CREATE TABLE IF NOT EXISTS keyword_cache (
  keyword TEXT NOT NULL,
  provider TEXT NOT NULL,
  results_json TEXT NOT NULL,
  fetched_at TEXT NOT NULL,
  PRIMARY KEY (keyword, provider)
);
"""

D1_FTS = """-- Optional full-text index for the local fallback search.
CREATE VIRTUAL TABLE IF NOT EXISTS ontology_search USING fts5(provider, acronym, name, description, categories);
"""


def d1_row(r: dict) -> dict:
    """One ontology_scores row for D1 from a results_frame record (shared with the nightly job)."""
    import math

    def g(k):
        v = r.get(k)
        if isinstance(v, float) and math.isnan(v):
            return None
        if hasattr(v, "item"):  # numpy scalar -> Python
            v = v.item()
            if isinstance(v, float) and math.isnan(v):
                return None
        return v

    def num(k, cast=float, nd=None):
        v = g(k)
        if v is None:
            return None
        v = cast(v)
        return round(v, nd) if nd is not None else v

    detail = g("eval_error") or g("download_error") or g("catalog_error")
    return {
        "provider": g("provider"), "acronym": g("acronym"), "name": g("name"),
        "description": (g("bp_description") or "")[:2000] or None, "categories": g("bp_categories"),
        "ontology_language": g("bp_language"),
        "submission_id": num("submission_id", int),
        "version": g("bp_version"), "released": g("bp_released"), "bioportal_url": g("bp_url"),
        "file_sha256": g("download_sha256"), "status": g("web_status"),
        "status_detail": str(detail)[:300] if detail else None,
        "describe_score": num("describe_score"), "define_score": num("define_score"),
        "define_bp_score": num("define_bp_score"), "define_bp_source": g("define_bp_source"),
        "connection_score": num("connection_score"), "flat_score": num("flat_score"),
        "core_average": num("core_average", float, 4), "core_average_bp": num("core_average_bp", float, 4),
        "logical_consistency_score": None, "structural_dist_score": None, "semantic_dist_score": None,
        "triple_count": num("triple_count", int), "entity_count": num("entities", int),
        "class_count": num("classes", int),
        "wiseowl_version": g("wiseowl_source_version"), "core4_version": g("core4_version"),
        "device": g("define_device"), "evaluated_at": g("eval_finished_at"), "updated_at": now(),
    }


def export_phase(ctx: Ctx) -> dict:
    """Write every table and summary file to exports/. Safe to run any time."""
    import pandas as pd
    exp = ctx.paths["exports"]
    df = results_frame(ctx)
    files: dict = {}

    full_csv = exp / "ontology_results_full.csv"
    df.to_csv(full_csv, index=False)
    files["ontology_results_full.csv"] = "Every ontology: BioPortal metadata, download, evaluation, all summary fields."
    try:
        df.to_parquet(exp / "ontology_results_full.parquet", index=False)
        files["ontology_results_full.parquet"] = "Same as the CSV, typed columns (needs pyarrow)."
    except Exception as exc:  # noqa: BLE001
        ctx.event("export", None, "WARNING", f"parquet skipped: {exc}")

    score_cols = ["provider", "acronym", "name", "web_status", "describe_score", "define_score",
                  "define_bp_score", "connection_score", "flat_score", "core_average", "core_average_2dp",
                  "core_average_bp", "core_average_bp_2dp", "define_bp_source", "weakest_metric",
                  "bp_categories", "bp_language", "submission_id", "bp_version", "bp_released", "bp_url"]
    df[[c for c in score_cols if c in df.columns]].to_csv(exp / "scores_compact.csv", index=False)
    files["scores_compact.csv"] = "One row per ontology with the four scores and web status."

    bench_cols = ["acronym", "eval_status", "eval_bytes", "triple_count", "entities", "classes",
                  "individuals", "bp_classes", "sniffed_format", "parse_format", "eval_seconds",
                  "wall_seconds", "parent_peak_rss_mb", "gpu_peak_allocated_mb", "define_device",
                  "define_defined"] + [c for c in df.columns if c.startswith("time_")] + [
                  "define_seconds_encode_labels", "define_seconds_encode_definitions", "download_seconds"]
    bench = df[[c for c in bench_cols if c in df.columns]]
    if "eval_status" in bench.columns:
        bench = bench[bench.eval_status.notna()]
    bench.to_csv(exp / "timings_and_memory.csv", index=False)
    files["timings_and_memory.csv"] = "Per-ontology size, stage timings, RAM and GPU use (the benchmark table)."

    fail_cols = ["acronym", "name", "web_status", "catalog_status", "catalog_error", "download_status",
                 "download_http_status", "download_error", "eval_status", "eval_error_type", "eval_error",
                 "eval_stage_at_failure", "eval_attempts", "eval_bytes", "bp_classes", "bp_language"]
    fails = df[df.web_status != "scored"]
    fails[[c for c in fail_cols if c in fails.columns]].to_csv(exp / "not_scored.csv", index=False)
    files["not_scored.csv"] = "Every ontology without scores and the reason."

    with open(exp / "results_detail.jsonl", "w", encoding="utf-8") as fh:
        for r in ctx.state.q("SELECT result_path FROM evaluations WHERE status='done' AND result_path IS NOT NULL"):
            p = Path(r["result_path"])
            if p.exists():
                fh.write(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False) + "\n")
    files["results_detail.jsonl"] = "Full nested result.json of each scored ontology, one per line."

    pd.DataFrame(ctx.state.q("SELECT * FROM events ORDER BY id")).to_csv(exp / "events.csv", index=False)
    files["events.csv"] = "Log of warnings, errors and phase summaries across all sessions."
    pd.DataFrame(ctx.state.q("SELECT * FROM runs ORDER BY started_at")).to_csv(exp / "sessions.csv", index=False)
    files["sessions.csv"] = "Every notebook session: config and environment."
    sb = ctx.state.q("SELECT * FROM search_benchmark ORDER BY id")
    if sb:
        pd.DataFrame(sb).to_csv(exp / "search_benchmark_all.csv", index=False)
        files["search_benchmark_all.csv"] = "BioPortal /search and /recommender latency per keyword."

    # data dictionary
    dd = []
    for col in df.columns:
        desc = COLUMN_DOCS.get(col)
        if desc is None:
            for pref, pdesc in PREFIX_DOCS.items():
                if col.startswith(pref):
                    desc = pdesc
                    break
        dd.append({"column": col, "dtype": str(df[col].dtype), "non_null": int(df[col].notna().sum()),
                   "description": desc or ""})
    pd.DataFrame(dd).to_csv(exp / "data_dictionary.csv", index=False)
    files["data_dictionary.csv"] = "Every column in ontology_results_full.csv with type, fill count, meaning."

    # D1 SQL
    (exp / "d1_schema.sql").write_text(D1_SCHEMA, encoding="utf-8")
    (exp / "d1_fts.sql").write_text(D1_FTS, encoding="utf-8")
    lines = []
    fts_lines = []
    for r in df.to_dict("records"):
        row = d1_row(r)
        lines.append(f"INSERT OR REPLACE INTO ontology_scores ({', '.join(row)}) VALUES "
                     f"({', '.join(_sql(v) for v in row.values())});")
        fts_lines.append("INSERT INTO ontology_search (provider, acronym, name, description, categories) VALUES "
                         f"({_sql(row['provider'])}, {_sql(row['acronym'])}, {_sql(row['name'])}, "
                         f"{_sql(row['description'])}, {_sql(row['categories'])});")
    (exp / "d1_data.sql").write_text("\n".join(lines) + "\n", encoding="utf-8")
    (exp / "d1_fts_data.sql").write_text("DELETE FROM ontology_search;\n" + "\n".join(fts_lines) + "\n", encoding="utf-8")
    files["d1_schema.sql"] = "CREATE TABLE statements for Cloudflare D1 (includes empty columns for the 3 later metrics)."
    files["d1_data.sql"] = "INSERT OR REPLACE rows for ontology_scores."
    files["d1_fts.sql"] = "Optional full-text search table (fallback search)."
    files["d1_fts_data.sql"] = "Rows for the full-text search table."

    # run summary
    scored = df[df.web_status == "scored"]
    summary = {
        "time": now(), "run_id": ctx.run_id, "notebook_version": NOTEBOOK_VERSION,
        "work_dir": str(ctx.work), "ontologies_in_catalog": int(len(df)),
        "web_status_counts": df.web_status.value_counts().to_dict(),
        "status": status(ctx),
        "score_stats": {c: core._stats(scored[c].dropna().to_numpy()) for c in
                        ("describe_score", "define_score", "define_bp_score", "connection_score", "flat_score",
                         "core_average", "core_average_bp")
                        if c in scored.columns},
        "config": ctx.cfg,
    }
    if "eval_seconds" in df.columns:
        summary["evaluation_seconds_total"] = float(df.eval_seconds.fillna(0).sum())
    core.dump_json(summary, str(exp / "run_summary.json"))
    files["run_summary.json"] = "Counts, score distributions, total time, config."

    readme = ["# Output files", "", f"Generated {now()} by {NOTEBOOK_VERSION}.", "",
              "## exports/", ""] + [f"- `{k}`: {v}" for k, v in files.items()] + [
        "", "## Other folders", "",
        "- `state.sqlite`: the resume database (catalog, downloads, evaluations, events, runs, search_benchmark).",
        "- `raw/<ACRONYM>/`: raw BioPortal JSON (ontology record, latest submission, metrics).",
        "- `raw/ontologies_list.json`: the full BioPortal listing at catalog time.",
        "- `downloads/<ACRONYM>/<submission>/`: downloaded files (and `extracted/` for zip archives).",
        "- `results/<ACRONYM>/<submission>/result.json`: full nested result for one ontology.",
        "- `results/<ACRONYM>/<submission>/entities.csv.gz`: one row per entity with every per-entity value.",
        "- `logs/pipeline.log`, `logs/worker.log`: text logs.",
        "- `environment/environment_<run>.json`: hardware and library versions per session.",
        "- `parity/`: WiseOwl test files and outputs used for the parity check.",
    ]
    (exp / "README_outputs.md").write_text("\n".join(readme) + "\n", encoding="utf-8")
    ctx.event("export", None, "INFO", f"exported {len(files)} files to {exp}")
    return {"export_dir": str(exp), "files": files, "web_status_counts": summary["web_status_counts"]}

## Start the session

Loads the modules, opens (or creates) the resume database, and records this machine's hardware and library versions. Check the GPU lines in the output.

In [ ]:
import sys, os, importlib, json
sys.path.insert(0, os.getcwd())
import owl4_core, owl4_worker, owl4_pipeline
for m in (owl4_core, owl4_worker, owl4_pipeline):
    importlib.reload(m)
P = owl4_pipeline

ctx = P.init(CONFIG, api_key=BIOPORTAL_API_KEY)
env = P.environment_report(ctx)

print("Work folder:     ", ctx.work)
print("Python:          ", env["python"].split()[0], "| torch", env.get("torch_version"), "| CUDA build", env.get("torch_cuda_build"))
print("RAM total / free:", env.get("ram_total_gb"), "GB /", env.get("ram_available_gb"), "GB | disk free", env.get("disk_free_gb"), "GB")
print("CUDA available:  ", env.get("cuda_available"))
if env.get("cuda_available"):
    print("GPU:             ", env.get("gpu_name"), "|", env.get("gpu_memory_gb"), "GB | capability", env.get("gpu_capability"))
    if not env.get("gpu_arch_supported_by_this_torch"):
        print("WARNING: this PyTorch build does not list your GPU architecture. Reinstall torch from the cu128 (or newer) index.")
else:
    print("WARNING: no CUDA GPU found. Define will run on the CPU (much slower). Check the torch install.")
print("nvidia-smi:      ", env.get("nvidia_smi"))
print("Status so far:   ", P.status(ctx))

## Step 1: Parity check

Downloads WiseOwl's two test ontologies (`tiny.owl`, `GoodRelations.owl`) and their stored correct scores from GitHub, runs them through the same background worker used for the real run, and compares. The first run also downloads the BERT model (about 440 MB), so it can take a few minutes.

Both lines must say `OK` before the full run.

In [ ]:
report = P.parity_check(ctx)
print("Worker device:", report["worker"]["device"], "|", report["worker"].get("gpu_name"))
for case in report["cases"]:
    print(f"{case['name']:<14} {'OK' if case['ok'] else 'MISMATCH'}  got={case.get('got')}  expected={case.get('expected')}  {case.get('error') or ''}")
print("All OK" if report["all_ok"] else "Parity FAILED: do not start the full run; check logs/worker.log")

## Step 2: Catalog

Lists every BioPortal ontology and stores its metadata, latest submission, and BioPortal's own metrics. About two API calls per ontology, so roughly 10 minutes at 5 requests per second. Re-running skips ontologies already fetched. Use `refresh=True` to fetch everything again.

In [ ]:
counts = P.catalog_phase(ctx, refresh=False)
cat = P.catalog_frame(ctx)
print(counts)
print(cat.catalog_status.value_counts().to_dict())
print("Viewing restriction:", cat.bp_viewing_restriction.value_counts(dropna=False).to_dict())
print("Ontology language:  ", cat.bp_language.value_counts(dropna=False).head(10).to_dict())
cat[["acronym", "name", "submission_id", "bp_language", "bp_classes", "bp_viewing_restriction"]].head(10)

### Optional: search latency benchmark

Times BioPortal's `/search` and `/recommender` endpoints on 40 sample keywords. This answers the question from the plan: which endpoint is fast enough for the 2 second web target. It does not affect the scores.

In [ ]:
bench = P.search_benchmark(ctx)   # or P.search_benchmark(ctx, ["diabetes", "cancer"])
print(bench.groupby("endpoint")["seconds"].describe(percentiles=[0.5, 0.9, 0.95]).round(2))

## Step 3: Pilot run (Phase 0 benchmark)

Picks about 20 ontologies spread over size bins by BioPortal class count (under 1,000; 1,000 to 20,000; 20,000 to 200,000; over 200,000), downloads and scores only those, and exports. Open `exports/timings_and_memory.csv` afterwards: it shows file size, triples, time per stage, RAM, and GPU memory for each one. Use it to set `eval_timeout_min` and `worker_ram_limit_gb` for the full run.

The pilot results count toward the full run, so nothing is repeated later.

In [ ]:
pilot = P.select_pilot(ctx, n=20)
print("Pilot:", pilot)
print(P.download_phase(ctx, only=pilot))
print(P.evaluate_phase(ctx, only=pilot))
P.export_phase(ctx)
import pandas as pd
t = pd.read_csv(ctx.paths["exports"] / "timings_and_memory.csv")
t[t.acronym.isin(pilot)].sort_values("eval_bytes")[
    [c for c in ["acronym", "eval_status", "eval_bytes", "triple_count", "entities", "time_parse", "time_index",
                 "time_define", "time_total", "parent_peak_rss_mb", "gpu_peak_allocated_mb"] if c in t.columns]]

## Step 4: Download everything

Downloads the latest submission of every ontology, smallest first. Private, summary-only, and license-restricted ontologies are recorded and skipped. Re-running skips files already on disk. `retry_failed=True` tries failed downloads again.

In [ ]:
print(P.download_phase(ctx, retry_failed=False))

## Step 5: Evaluate everything

Scores every downloaded ontology, smallest file first, one line per ontology. This is the long step.

- **To pause:** use Kernel, Interrupt. The ontology in progress goes back to the queue.
- **To resume:** run the setup cells (Settings, API key, the three code-module cells, Start the session), then this cell again.
- **After a crash or power loss:** same as resume. An ontology that was running when the session died is retried, up to `max_attempts`. An ontology that crashes the worker every time ends as `gave_up`, so it cannot block the run.

In [ ]:
print(P.evaluate_phase(ctx))

In [ ]:
# Progress at any time (also works while nothing is running)
import pandas as pd
print(json.dumps(P.status(ctx), indent=1))
ev = pd.DataFrame(ctx.state.q("SELECT acronym, status, attempts, seconds, core_average, error FROM evaluations"))
ev.sort_values("seconds", ascending=False).head(15) if not ev.empty else ev

## Step 6: Repair failures (new in 1.1)

Works on any run, including your version 1.0 folder. It does three things, in order:

1. Retries downloads that failed (for example PR, which broke part-way in 1.0).
2. Re-evaluates parse failures and empty results with the 1.1 parser (owlready2 fallback, not-an-ontology detection).
3. For anything still failing or empty, tries other sources: BioPortal's original upload, then for OBO ontologies the OWL file from the OBO Foundry (for example CHEBI). An OWL file from the OBO Foundry may be a newer version than BioPortal's submission; the `download_variant` column shows `obo_owl:...` for those.

The first cell only shows the plan and changes nothing.

In [ ]:
plan = P.plan_v11(ctx)
for key in ("download_failed", "parse_failed", "empty", "define_candidates", "flat_recheck"):
    print(f"{key:<18} {len(plan[key]):>4}  {plan[key][:25]}{' ...' if len(plan[key]) > 25 else ''}")
print("already on 1.1:   ", len(plan["already_v11"]))

In [ ]:
repair_report = P.repair_phase(ctx)

### Step 6b: Get very large OBO ontologies from the OBO Foundry

BioPortal cut off the PR download at about 1 GB on every attempt, and the GAZ file from the OBO Foundry broke part-way once. This cell fetches both OWL files from the OBO Foundry and scores them. Each one is skipped if it already has a score, so the cell is safe to re-run. The `download_variant` column shows `url:purl.obolibrary.org` for these, because the OBO Foundry file may be a newer version than BioPortal's submission.

In [ ]:
for acronym, url in [("PR", "http://purl.obolibrary.org/obo/pr.owl"),
                     ("GAZ", "http://purl.obolibrary.org/obo/gaz.owl")]:
    P.fetch_from_url(ctx, acronym, url)

## Step 7: Re-score with the BioPortal-aware Define (new in 1.1)

Re-scores only the ontologies where version 1.1 can change a result:

- BioPortal lists a definition property that WiseOwl does not read, or BioPortal counts at least twice as many definitions as WiseOwl found (NCIT, ORDO, RadLex, and others);
- the ontology has subclass cycles and a shallow taxonomy, so Flat could have moved by 1 point in 1.0.

It also re-scores ontologies whose BioPortal-aware Define was computed by an earlier version (1.1.1 ignores `rdfs:isDefinedBy` and link values). Every other ontology keeps its 1.0 scores, and its BioPortal-aware Define equals its strict Define. To re-score everything instead (about as long as the first run), use `P.define_bp_phase(ctx, full=True)`.

In [ ]:
print(P.define_bp_phase(ctx))

## Step 8 (optional): Retry failures with bigger limits

After looking at `exports/not_scored.csv`, you can put some failures back in the queue, raise a limit, and run Step 5 again. Examples below are commented out.

In [ ]:
# Retry everything that timed out, with a 6 hour limit:
# P.requeue(ctx, statuses=("timeout",)); ctx.cfg["eval_timeout_min"] = 360; print(P.evaluate_phase(ctx))

# Retry two specific ontologies:
# P.requeue(ctx, statuses=("timeout", "memory_limit", "gave_up", "error", "crashed"), acronyms=["NCIT", "CHEBI"])

# Retry failed downloads:
# print(P.download_phase(ctx, retry_failed=True))

# Score a local file you already have (goes through the same worker and exports):
# P.add_local_file(ctx, r"C:\path\to\file.owl", acronym="MYONTO"); print(P.evaluate_phase(ctx, only=["MYONTO"]))

## Step 9: Export

Writes every table and summary to `exports/`. Safe to run at any time, including mid-run. The file `exports/README_outputs.md` describes every output file, and `exports/data_dictionary.csv` describes every column.

In [ ]:
out = P.export_phase(ctx)
print("Export folder:", out["export_dir"])
for name, desc in out["files"].items():
    print(f"  {name:<34} {desc}")
print("Web status counts:", out["web_status_counts"])

# Where the BioPortal-aware Define differs from the strict one
import pandas as pd
sc = pd.read_csv(ctx.paths["exports"] / "scores_compact.csv")
changed = sc[(sc.web_status == "scored") & (sc.define_bp_score != sc.define_score)]
print(f"\n{len(changed)} ontologies where the BioPortal-aware Define differs:")
changed.sort_values("define_bp_score", ascending=False)[
    ["acronym", "define_score", "define_bp_score", "core_average_2dp", "core_average_bp_2dp", "define_bp_source"]]

In [ ]:
# Quick look at the score distributions (needs matplotlib; skip if not installed)
import pandas as pd
df = pd.read_csv(ctx.paths["exports"] / "scores_compact.csv")
scored = df[df.web_status == "scored"]
print(scored[["describe_score", "define_score", "define_bp_score", "connection_score", "flat_score",
              "core_average", "core_average_bp"]].describe().round(2))
try:
    import matplotlib.pyplot as plt
    ax = scored[["describe_score", "define_score", "define_bp_score", "connection_score", "flat_score",
                 "core_average_bp"]].hist(bins=20, figsize=(12, 7))
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed: pip install matplotlib to see histograms")

## What gets recorded

| Where | What |
|---|---|
| `state.sqlite` | Resume database: catalog, downloads, evaluations, events, sessions, search benchmark |
| `raw/<ACRONYM>/` | Raw BioPortal JSON: ontology record, latest submission, BioPortal metrics |
| `downloads/<ACRONYM>/<submission>/` | Downloaded file, plus `extracted/` for zip archives |
| `results/<ACRONYM>/<submission>/result.json` | Everything about one ontology: scores, every metric's internals, parse attempts, ontology statistics, timings, memory |
| `results/<ACRONYM>/<submission>/entities.csv.gz` | One row per class or individual: label, label source, Describe signals, definition length, BERT tokens, cosine, match, adequacy, Define value, connection links, taxonomy children (plus the BioPortal-aware Define value where it differs) |
| `results/<ACRONYM>/<submission>/parse_failure.json` | For files that could not be parsed: every format tried and its error |
| `exports/ontology_results_full.csv` (and `.parquet`) | One row per ontology with all BioPortal fields, download info, evaluation status, and every summary value |
| `exports/scores_compact.csv` | The four scores, average, and web status |
| `exports/timings_and_memory.csv` | Size, per-stage time, RAM, and GPU memory per ontology |
| `exports/not_scored.csv` | Every ontology without scores and why |
| `exports/results_detail.jsonl` | All `result.json` files in one file |
| `exports/d1_*.sql` | Schema and rows for Cloudflare D1 (with empty columns for the three later metrics) |
| `exports/data_dictionary.csv`, `README_outputs.md` | Meaning of every column and file |
| `logs/pipeline.log`, `logs/worker.log` | Text logs |
| `environment/` | Hardware and library versions for each session |

**Notes on the scores.** `define_score` and `core_average` are strict WiseOwl. `define_bp_score` and `core_average_bp` also read the ontology's own definition property; the web page uses these. WiseOwl reads one file and does not follow `owl:imports`, so imported content is not scored; `owl_imports_count` records how many imports each ontology declares. WiseOwl also counts deprecated classes as entities; `deprecated_entities` records how many there are.